# Credit Migration Rate Sensitivity Model

Comparing sensitivity of corporate earnings in relation to the 10-Year US Treasury across credit tiers.

This study shows how a variation within the 10 Year US treasury, coupled with the different credit tranches demonstrates financial resilience during high rates environments. In this study, I first set up some ground truths on statistical values such as Beta hat as well as assumptions on yields and credit tranches, as shown later.

I then generated a macro feature simulation to simlate the 10 Year Treasury moves versus the corporate earning moves. While the Markov DGP predicts Beta hat with a 4x4 matrix, representing the 4 credit tranches and their demontions/promotions/unchanged.

The feature engineering section features the train test split to test the model on the full historical data P hat t. Applying hthe log odds ratio (Y) to transfer standard probabilities of 0-100% into unbounded values between positive and negative infinity to saftly run the WLS model without generating impossible prediction. Then implemented a design matrix builder.

The WLS is then applied with the equation Beta hat connecting X and Y. We then check if the engine is accurate by testing if Beta hat is approximately equal to Beta True.

Using the Markov engine, the softmax forcasts predicts the decrease of credit rating compared to the rating staying constant, formatted in a 4x4 table. We then continue to use softmax to validate the sochastic probability matrices. Finally stretching into n quarters, creating a cummulative probability and making an evaluation with statistical tests.



In [ ]:
import numpy as np
import pandas as pd
import json
import math

Assumptions: These are ground truths plugged in as the basis of the study.
1. Towards the yield, such as the amplitude and idiosyncratic shock
2. The nature of the credit traches especially on the assumption on Beta True and the number of companies per tranch
3. The movement of earnings like the earnings sensitivity to yields and the idiosyncratic moves.



In [ ]:
class config:

    N_QUARTERS = 60
    N_BURNIN = 20
    TRAIN_FRACTION = 0.80
    N_TRAIN = int(round(N_QUARTERS * TRAIN_FRACTION))
    N_TEST = N_QUARTERS - N_TRAIN

    RANDOM_SEED = 42

    YIELD_START = 0.035
    YIELD_CYCLE_BASE = 0.035
    YIELD_CYCLE_AMPLITUDE = 0.012
    YIELD_CYCLE_PERIOD_QUARTERS = 32
    YIELD_MEAN_REVERSION_SPEED = 0.35
    YIELD_SHOCK_STD = 0.0009

    EARNINGS_START = 0.015
    EARNINGS_LONG_RUN_MEAN = 0.015
    EARNINGS_MEAN_REVERSION_SPEED = 0.30
    EARNINGS_YIELD_SENSITIVITY = -2.5
    EARNINGS_SHOCK_STD = 0.0012
    OUTLIER_Z_THRESHOLD = 3.0


    TRANCHES = ["IG", "NonIG", "Junk", "Default"]
    N_TRANCHES = len(TRANCHES)
    DEFAULT_IDX = 3

    MAX_FALL = {"IG": 3, "NonIG": 2, "Junk": 1}

    CASCADE_STAGES = {"IG": ["A", "B", "C"], "NonIG": ["A", "B"], "Junk": ["A"]}

    UPGRADE_PROB = {"IG": 0.0, "NonIG": 0.04, "Junk": 0.05}


    BETA_TRUE = {
        "IG":    {"A": np.array([-5.4, -1.3, 1.0, -0.4]),
                  "B": np.array([-1.8, -0.5, 0.6, -0.15]),
                  "C": np.array([-2.3, -0.4, 0.5, -0.1])},
        "NonIG": {"A": np.array([-4.2, -1.1, 1.1, -0.35]),
                  "B": np.array([-1.3, -0.45, 0.65, -0.15])},
        "Junk":  {"A": np.array([-2.6, -0.9, 1.25, -0.3])},
    }

    FEATURE_NAMES = ["X0_intercept", "X1_delta_earnings", "X2_delta_yield", "X3_interaction"]
    COMPANY_POOL_SIZE = 3000
    SMOOTHING_EPSILON = 0.5
    EXTRAPOLATION_CAP_X1_MULTIPLE = 3.0  # earnings: 3x its own training range
    EXTRAPOLATION_CAP_X2_BOUNDS = (-2.0, 10.0)  # yield: explicit range, percentage points
    EXTRAPOLATION_CAP_X3_MULTIPLE = 15.0  # interaction: 15x its own training range
    WLS_WEIGHT_CAP_MULTIPLE = 5.0

## Step 1a: Macro feature simulation

The simulation helps generate the explanatory variables, in short, the simulation will come to replace actual data that can be extracted by this set of synthetic data in which the study would be based on.

In [ ]:
def simulate_macro_paths(seed: int = config.RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    total_periods = config.N_BURNIN + config.N_QUARTERS
    yield_level = np.empty(total_periods + 1)
    earnings_growth = np.empty(total_periods + 1)
    yield_level[0] = config.YIELD_START
    earnings_growth[0] = config.EARNINGS_START

    yield_shocks = rng.normal(0.0, config.YIELD_SHOCK_STD, size=total_periods)
    earnings_shocks = rng.normal(0.0, config.EARNINGS_SHOCK_STD, size=total_periods)


    t_idx = np.arange(total_periods)
    cycle_target = config.YIELD_CYCLE_BASE + config.YIELD_CYCLE_AMPLITUDE * np.sin(
        2 * np.pi * t_idx / config.YIELD_CYCLE_PERIOD_QUARTERS
    )

    for t in range(total_periods):
        dy = config.YIELD_MEAN_REVERSION_SPEED * (cycle_target[t] - yield_level[t]) \
             + yield_shocks[t]
        yield_level[t + 1] = yield_level[t] + dy

        de = config.EARNINGS_MEAN_REVERSION_SPEED * (config.EARNINGS_LONG_RUN_MEAN - earnings_growth[t]) \
             + config.EARNINGS_YIELD_SENSITIVITY * dy \
             + earnings_shocks[t]
        earnings_growth[t + 1] = earnings_growth[t] + de


    yield_level = yield_level[config.N_BURNIN:]
    earnings_growth = earnings_growth[config.N_BURNIN:]

    delta_yield = np.diff(yield_level)
    delta_earnings = np.diff(earnings_growth)

    df = pd.DataFrame({
        "quarter": np.arange(1, config.N_QUARTERS + 1),
        "yield_level": yield_level[1:],
        "earnings_growth_level": earnings_growth[1:],
        "delta_yield": delta_yield,
        "delta_earnings": delta_earnings,
    })
    return df


def noise_budget_report(df: pd.DataFrame) -> dict:
    report = {}
    for col, shock_std, series_name in [
        ("delta_yield", config.YIELD_SHOCK_STD, "yield"),
        ("delta_earnings", config.EARNINGS_SHOCK_STD, "earnings"),
    ]:
        series = df[col].to_numpy()
        total_std = series.std(ddof=1)
        # idiosyncratic-shock-std / total-std: lower = signal-dominated, as intended
        noise_ratio = shock_std / total_std if total_std > 0 else np.nan
        z = (series - series.mean()) / total_std if total_std > 0 else np.zeros_like(series)
        outlier_mask = np.abs(z) > config.OUTLIER_Z_THRESHOLD
        report[series_name] = {
            "idiosyncratic_shock_std": shock_std,
            "realized_total_std": total_std,
            "noise_to_total_std_ratio": noise_ratio,
            "n_outlier_quarters": int(outlier_mask.sum()),
            "outlier_quarters": df.loc[outlier_mask, "quarter"].tolist(),
            "min": series.min(),
            "max": series.max(),
        }
    return report

In [ ]:
macro_df = simulate_macro_paths()
noise_report = noise_budget_report(macro_df)
print(macro_df.head())
print()
for series_name, r in noise_report.items():
    print(f"{series_name}: idiosyncratic_std={r['idiosyncratic_shock_std']:.4f} "
          f"realized_std={r['realized_total_std']:.4f} "
          f"noise/total_ratio={r['noise_to_total_std_ratio']:.3f} "
          f"outliers(|z|>{config.OUTLIER_Z_THRESHOLD})={r['n_outlier_quarters']}")

Design Matrix builder

The builder ensures that the result of Beta hat is directly comparable with the results of Beta true, found within the assumptions section governing the study. This builder would be used by the Markov DGP and Feature engineering later during further analysis


In [ ]:
def build_feature_matrix(macro_df: pd.DataFrame) -> np.ndarray:
    x0 = np.ones(len(macro_df))
    x1 = macro_df["delta_earnings"].to_numpy() * 100.0
    x2 = macro_df["delta_yield"].to_numpy() * 100.0
    x3 = x1 * x2
    return np.column_stack([x0, x1, x2, x3])

## Step 1b: Markov data-generating process

By using the cascade methodology to build 1 tranche's row of the transition matrix of possible movement of the company from a certain delta in yield. for 1 quarter and the sigmoid to ensure the log odd function is converted into probabilities from feature enginerring which protects the model against outlier situations.

In [ ]:
def _sigmoid(z: float) -> float:

    if z >= 0:
        return 1.0 / (1.0 + np.exp(-z))
    exp_z = np.exp(z)
    return exp_z / (1.0 + exp_z)


def cascade_row(tranche: str, x_row: np.ndarray, beta_stage_dict: dict) -> np.ndarray:

    idx = config.TRANCHES.index(tranche)
    u = config.UPGRADE_PROB[tranche]
    max_fall = config.MAX_FALL[tranche]
    betas = beta_stage_dict[tranche]

    logit_a = float(x_row @ betas["A"])
    p_fall_ge1 = _sigmoid(logit_a)
    p_stay = 1.0 - p_fall_ge1

    p_fall_eq = {}
    if max_fall == 1:
        p_fall_eq[1] = p_fall_ge1
    else:
        logit_b = float(x_row @ betas["B"])
        p_fall_ge2_given = _sigmoid(logit_b)
        p_fall_eq[1] = p_fall_ge1 * (1.0 - p_fall_ge2_given)
        p_fall_ge2 = p_fall_ge1 * p_fall_ge2_given
        if max_fall == 2:
            p_fall_eq[2] = p_fall_ge2
        else:  # max_fall == 3
            logit_c = float(x_row @ betas["C"])
            p_eq3_given = _sigmoid(logit_c)
            p_fall_eq[2] = p_fall_ge2 * (1.0 - p_eq3_given)
            p_fall_eq[3] = p_fall_ge2 * p_eq3_given

    row = np.zeros(config.N_TRANCHES)
    if u > 0:
        row[idx - 1] = u
    row[idx] += (1.0 - u) * p_stay
    for k, p in p_fall_eq.items():
        row[idx + k] += (1.0 - u) * p
    return row


def build_transition_tensor(feature_matrix: np.ndarray, beta_stage_dict: dict) -> np.ndarray:

    n = feature_matrix.shape[0]
    tensor = np.zeros((n, config.N_TRANCHES, config.N_TRANCHES))
    for t in range(n):
        x_row = feature_matrix[t]
        for tranche in ["IG", "NonIG", "Junk"]:
            idx = config.TRANCHES.index(tranche)
            tensor[t, idx, :] = cascade_row(tranche, x_row, beta_stage_dict)
        tensor[t, config.DEFAULT_IDX, config.DEFAULT_IDX] = 1.0  # absorbing
    return tensor


def simulate_observed_counts(p_true_tensor: np.ndarray, seed: int = config.RANDOM_SEED) -> np.ndarray:

    rng = np.random.default_rng(seed + 1)  # distinct stream from macro shocks
    n = p_true_tensor.shape[0]
    counts = np.zeros((n, config.N_TRANCHES, config.N_TRANCHES), dtype=int)
    for t in range(n):
        for tranche in ["IG", "NonIG", "Junk"]:
            idx = config.TRANCHES.index(tranche)
            probs = p_true_tensor[t, idx, :]
            probs = probs / probs.sum()  # guard against float drift
            counts[t, idx, :] = rng.multinomial(config.COMPANY_POOL_SIZE, probs)
        counts[t, config.DEFAULT_IDX, config.DEFAULT_IDX] = config.COMPANY_POOL_SIZE
    return counts


def counts_to_frequencies(counts: np.ndarray) -> np.ndarray:
    pool = counts.sum(axis=2, keepdims=True)
    return counts / pool

In [ ]:
X = build_feature_matrix(macro_df)
p_true = build_transition_tensor(X, config.BETA_TRUE)
counts = simulate_observed_counts(p_true)

for tranche in ['IG', 'NonIG', 'Junk']:
    idx = config.TRANCHES.index(tranche)
    p_fall_any = 1 - p_true[:, idx, idx] - (p_true[:, idx, idx-1] if idx > 0 else 0)
    print(f"{tranche}: P(any fall) mean={p_fall_any.mean():.4f} min={p_fall_any.min():.4f} "
          f"max={p_fall_any.max():.4f} | P(straight to default) max={p_true[:, idx, config.DEFAULT_IDX].max():.4f}")

## Step 2: feature engineering

Using the 80/20 train test model chronologically, to avoid data leakage to test P hat t, where the first 48 quarters were train and the last 12 as tests.

Then we apply the log odd split, using the formula and additive methods such as adding an error component to ensure undefined answer cannot occur, we convert probabilities bounded by 0-100% into unbounded probabilities to run the WLS model later.

Finally we apply the WLS, it is used to measure the amount of observation and its reliability within the study. This would be demonstrated int he next step.

In [ ]:
def train_test_indices():
    train_idx = np.arange(0, config.N_TRAIN)
    test_idx = np.arange(config.N_TRAIN, config.N_QUARTERS)
    return train_idx, test_idx


def _stage_log_odds_and_weight(counts_row: np.ndarray, tranche_idx: int, stage: str, max_fall: int):
    eps = config.SMOOTHING_EPSILON
    if stage == "A":
        n_stay = counts_row[tranche_idx] + eps
        n_fall = counts_row[tranche_idx + 1: tranche_idx + 1 + max_fall].sum() + eps
    elif stage == "B":
        n_stay = counts_row[tranche_idx + 1] + eps
        n_fall = counts_row[tranche_idx + 2: tranche_idx + 1 + max_fall].sum() + eps
    elif stage == "C":
        n_stay = counts_row[tranche_idx + 2] + eps
        n_fall = counts_row[tranche_idx + 3] + eps
    else:
        raise ValueError(f"unknown stage {stage}")

    y = float(np.log(n_fall / n_stay))
    variance = 1.0 / n_fall + 1.0 / n_stay
    weight = float(1.0 / variance)
    return y, weight


def build_Y_and_W_by_stage(counts: np.ndarray, indices: np.ndarray):

    Y, W = {}, {}
    for tranche in ["IG", "NonIG", "Junk"]:
        idx = config.TRANCHES.index(tranche)
        max_fall = config.MAX_FALL[tranche]
        Y[tranche], W[tranche] = {}, {}
        for stage in config.CASCADE_STAGES[tranche]:
            pairs = [
                _stage_log_odds_and_weight(counts[t, idx, :], idx, stage, max_fall)
                for t in indices
            ]
            Y[tranche][stage] = np.array([p[0] for p in pairs])
            w = np.array([p[1] for p in pairs])
            # Winsorize: a single near-zero-count quarter can otherwise get
            # a wildly unstable weight and dominate the fit -- see config.py.
            cap = config.WLS_WEIGHT_CAP_MULTIPLE * np.median(w)
            W[tranche][stage] = np.minimum(w, cap)
    return Y, W


def build_all_features(macro_df, counts: np.ndarray):

    X = build_feature_matrix(macro_df)
    train_idx, test_idx = train_test_indices()

    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, W_train = build_Y_and_W_by_stage(counts, train_idx)
    Y_test, W_test = build_Y_and_W_by_stage(counts, test_idx)

    return {
        "X_train": X_train, "X_test": X_test,
        "Y_train": Y_train, "Y_test": Y_test,
        "W_train": W_train, "W_test": W_test,
        "train_idx": train_idx, "test_idx": test_idx,
    }

In [ ]:
feats = build_all_features(macro_df, counts)
print('X_train:', feats['X_train'].shape, ' X_test:', feats['X_test'].shape)
print(f"train quarters: 1-{config.N_TRAIN}  test quarters: {config.N_TRAIN+1}-{config.N_QUARTERS}")
print('condition number of raw X_train\'X_train:', np.linalg.cond(feats['X_train'].T @ feats['X_train']))

## Step 3: OLS / WLS engine

We estimate Beta hat which measures how sensitive a downgrade is in terms of the delta of yield and earnings. We used WLS, which is a variant of OLS to give a weight of each quarter and how reliable it is, where quarters with more observed event is more important.

In [ ]:
def fit_ols(X: np.ndarray, Y: np.ndarray, weights: np.ndarray = None) -> dict:
    n, k = X.shape
    w = np.ones(n) if weights is None else np.asarray(weights, dtype=float)
    sqrt_w = np.sqrt(w)

    # Standard WLS-via-OLS trick: reweight both sides, then it's an ordinary
    # least-squares problem again -- still the same closed-form solve.
    Xw = X * sqrt_w[:, None]
    Yw = Y * sqrt_w

    s = np.linalg.norm(Xw, axis=0)
    s[s == 0] = 1.0  # guard, not expected to trigger
    Xw_scaled = Xw / s

    XtX_scaled = Xw_scaled.T @ Xw_scaled
    XtY_scaled = Xw_scaled.T @ Yw
    beta_scaled = np.linalg.solve(XtX_scaled, XtY_scaled)
    beta_hat = beta_scaled / s

    # Cross-check against SVD-based lstsq, on the SAME weighted/reweighted
    # data -- comparing a weighted solve against an unweighted lstsq would
    # always show a spurious "disagreement" that isn't actually a bug.
    beta_lstsq, *_ = np.linalg.lstsq(Xw, Yw, rcond=None)
    max_disagreement = float(np.max(np.abs(beta_hat - beta_lstsq)))

    Y_hat = X @ beta_hat  # fitted values in the original (unweighted) space
    residuals = Y - Y_hat
    weighted_residuals = sqrt_w * residuals
    sse = float(weighted_residuals @ weighted_residuals)

    # R^2 here is on the WEIGHTED scale (weighted SSE / weighted SST), the
    # standard convention for WLS. NOT directly comparable to a plain-OLS R^2
    # computed elsewhere (or to this same pipeline's R^2 before WLS was
    # introduced) even though both are called "R^2" -- say so explicitly
    # wherever these numbers get quoted.
    y_bar_w = float(np.sum(w * Y) / np.sum(w))
    sst = float(np.sum(w * (Y - y_bar_w) ** 2))
    r_squared = 1.0 - sse / sst if sst > 0 else np.nan

    dof = n - k
    sigma2 = sse / dof if dof > 0 else np.nan
    XtX_scaled_inv = np.linalg.inv(XtX_scaled)
    var_beta = sigma2 * (XtX_scaled_inv / np.outer(s, s))
    se_beta = np.sqrt(np.diag(var_beta))
    t_stats = beta_hat / se_beta

    return {
        "beta_hat": beta_hat,
        "Y_hat": Y_hat,
        "residuals": residuals,
        "r_squared": r_squared,
        "se_beta": se_beta,
        "t_stats": t_stats,
        "dof": dof,
        "weighted": weights is not None,
        "condition_number_raw": float(np.linalg.cond(X.T @ X)),
        "condition_number_preconditioned": float(np.linalg.cond(XtX_scaled)),
        "lstsq_max_disagreement": max_disagreement,
    }


def validate_wls_reduces_to_ols(X: np.ndarray, Y: np.ndarray) -> dict:
    """
    WLS with uniform weights (all 1s) must exactly reproduce plain OLS.
    Checked explicitly -- like validate_softmax_and_chain in the Markov
    engine -- rather than trusted by algebra alone.
    """
    plain = fit_ols(X, Y, weights=None)
    weighted_uniform = fit_ols(X, Y, weights=np.ones(len(Y)))
    max_diff = float(np.max(np.abs(plain["beta_hat"] - weighted_uniform["beta_hat"])))
    return {"max_beta_diff": max_diff, "passed": bool(max_diff < 1e-8)}


def monte_carlo_wls_vs_ols(p_true_tensor: np.ndarray, X_train: np.ndarray, train_idx: np.ndarray, n_seeds: int = 40) -> dict:
    """
    Is WLS actually a better estimator than plain OLS, or did a single
    sampling draw just happen to favor one of them for a given stage? Since
    this is a synthetic study, that question is directly answerable: hold
    the macro path (and therefore Ptrue) fixed, redraw the finite-sample
    company-pool noise n_seeds times, and compare each estimator's average
    cosine similarity to the known beta_true. A single-seed comparison (as
    used everywhere else in this pipeline, seed=config.RANDOM_SEED) can be
    misleading for the sparsest stages (few observed events -> high
    seed-to-seed variance in which estimator "wins") -- this check is what
    actually settles it.
    """
    scores = {tr: {st: {"ols": [], "wls": []} for st in stages} for tr, stages in config.CASCADE_STAGES.items()}
    for i in range(n_seeds):
        counts = simulate_observed_counts(p_true_tensor, seed=i * 1000 + 7)
        Y_train, W_train = build_Y_and_W_by_stage(counts, train_idx)
        for tranche, stages in config.CASCADE_STAGES.items():
            for stage in stages:
                y = Y_train[tranche][stage]
                w = W_train[tranche][stage]
                bt = config.BETA_TRUE[tranche][stage]
                for key, weights in (("ols", None), ("wls", w)):
                    bh = fit_ols(X_train, y, weights=weights)["beta_hat"]
                    cos_sim = float(bh @ bt / (np.linalg.norm(bh) * np.linalg.norm(bt)))
                    scores[tranche][stage][key].append(cos_sim)

    summary = {}
    for tranche, stages in scores.items():
        summary[tranche] = {}
        for stage, vals in stages.items():
            ols_arr, wls_arr = np.array(vals["ols"]), np.array(vals["wls"])
            summary[tranche][stage] = {
                "ols_mean_cos_sim": float(ols_arr.mean()), "ols_std": float(ols_arr.std()),
                "wls_mean_cos_sim": float(wls_arr.mean()), "wls_std": float(wls_arr.std()),
                "wls_wins": bool(wls_arr.mean() > ols_arr.mean()),
                "difference_within_one_std": bool(abs(wls_arr.mean() - ols_arr.mean()) < max(ols_arr.std(), wls_arr.std())),
            }
    return summary


def fit_all_stages(X_train: np.ndarray, Y_train: dict, W_train: dict = None) -> dict:
    """Runs fit_ols once per (tranche, stage) -- 6 regressions total. WLS if W_train given."""
    results = {}
    for tranche, stages in Y_train.items():
        results[tranche] = {}
        for stage, y in stages.items():
            w = W_train[tranche][stage] if W_train is not None else None
            results[tranche][stage] = fit_ols(X_train, y, weights=w)
    return results


def beta_hat_vs_true(fit_results: dict, beta_true: dict = config.BETA_TRUE) -> dict:
    comparison = {}
    for tranche, stages in fit_results.items():
        comparison[tranche] = {}
        for stage, res in stages.items():
            bh = res["beta_hat"]
            bt = beta_true[tranche][stage]
            abs_err = np.abs(bh - bt)
            # bt components are never 0 in the current calibration, but guard
            # anyway rather than silently emitting inf if that ever changes.
            rel_err = np.where(bt != 0, abs_err / np.where(bt != 0, np.abs(bt), 1.0), np.nan)
            cos_sim = float(bh @ bt / (np.linalg.norm(bh) * np.linalg.norm(bt)))
            comparison[tranche][stage] = {
                "beta_true": bt, "beta_hat": bh,
                "abs_error": abs_err, "rel_error": rel_err,
                "cosine_similarity": cos_sim,
            }
    return comparison


def engine_sanity_check(feature_matrix: np.ndarray, p_true_tensor: np.ndarray) -> dict:
    """
    Verifies the OLS engine + cascade math are correct in isolation from
    sampling noise: builds Y directly from the exact (noiseless) Ptrue
    probabilities instead of noisy observed counts, fits plain OLS (no
    weights -- there's no sampling noise to weight against here), and
    confirms beta_hat recovers beta_true almost exactly (up to float
    precision). This isolates "does the engine work" from "how much does
    finite-sample noise degrade the estimate", which real Phat-based fits
    will show.
    """
    results = {}
    for tranche in ["IG", "NonIG", "Junk"]:
        idx = config.TRANCHES.index(tranche)
        max_fall = config.MAX_FALL[tranche]
        results[tranche] = {}
        for stage in config.CASCADE_STAGES[tranche]:
            y_vals = []
            for t in range(feature_matrix.shape[0]):
                row = p_true_tensor[t, idx, :]
                if stage == "A":
                    p_stay = row[idx]
                    p_fall = row[idx + 1: idx + 1 + max_fall].sum()
                    y_vals.append(np.log(p_fall / p_stay))
                elif stage == "B":
                    p_eq1 = row[idx + 1]
                    p_ge2 = row[idx + 2: idx + 1 + max_fall].sum()
                    y_vals.append(np.log(p_ge2 / p_eq1))
                elif stage == "C":
                    y_vals.append(np.log(row[idx + 3] / row[idx + 2]))
            fit = fit_ols(feature_matrix, np.array(y_vals))
            results[tranche][stage] = {
                "beta_true": config.BETA_TRUE[tranche][stage],
                "beta_hat_noiseless": fit["beta_hat"],
                "max_abs_diff": float(np.max(np.abs(fit["beta_hat"] - config.BETA_TRUE[tranche][stage]))),
            }
    return results

In [ ]:
print('=== WLS-reduces-to-OLS check ===')
print(validate_wls_reduces_to_ols(feats['X_train'], feats['Y_train']['IG']['A']))

print()
print('=== Engine sanity check (noiseless Ptrue) ===')
sanity = engine_sanity_check(X, p_true)
for tranche, stages in sanity.items():
    for stage, r in stages.items():
        print(f"{tranche}-{stage}: max|beta_hat-beta_true| = {r['max_abs_diff']:.9f}")

print()
print('=== Monte Carlo: WLS vs OLS across 40 independent sampling draws ===')
mc = monte_carlo_wls_vs_ols(p_true, feats['X_train'], feats['train_idx'], n_seeds=40)
for tranche, stages in mc.items():
    for stage, r in stages.items():
        verdict = 'WLS wins' if r['wls_wins'] else 'OLS wins'
        if r['difference_within_one_std']:
            verdict += ' (within 1 std -- toss-up, not a real difference)'
        print(f"{tranche}-{stage}: OLS={r['ols_mean_cos_sim']:.3f}+/-{r['ols_std']:.3f}  "
              f"WLS={r['wls_mean_cos_sim']:.3f}+/-{r['wls_std']:.3f}  {verdict}")

print()
print('=== Real WLS fit on noisy Phat (train only) ===')
fits = fit_all_stages(feats['X_train'], feats['Y_train'], feats['W_train'])
comparison = beta_hat_vs_true(fits)
for tranche, stages in fits.items():
    for stage, res in stages.items():
        c = comparison[tranche][stage]
        print(f"{tranche}-{stage}: R2={res['r_squared']:.3f} cos_sim={c['cosine_similarity']:.3f} "
              f"dof={res['dof']}")
        print(f"    beta_true={np.round(c['beta_true'],3)}  beta_hat={np.round(c['beta_hat'],3)}")
        print(f"    SE(beta_hat)={np.round(res['se_beta'],3)}  t_stats={np.round(res['t_stats'],2)}")

beta_hat = {tr: {st: fits[tr][st]['beta_hat'] for st in stages} for tr, stages in feats['Y_train'].items()}

## Step 4: Markov engine, prediction, evaluation, export

Softmax/chain primitives validated against a known 2-state case first. Multi-period cumulative probability uses sequential matrix multiplication `P(t1) @ P(t2) @ ... @ P(tn)` (non-homogeneous chain -- not a matrix power). Evaluated against `Ptrue` on the held-out test quarters (its only use anywhere in the pipeline). Also computes the observed X_train range and flags (without blocking) any forecast queried outside it. `predict_scenario` is the exploratory "what if" entry point (what Sheets/Excel call): it additionally CAPS extreme inputs before the sigmoid math runs, using PER-FEATURE bounds (not one shared multiplier) -- earnings and the interaction term are multiples of their own training range, yield is an explicit literal range, chosen from where the fitted model is actually still informative rather than derived mechanically. Confirmed with the user; fixes a real issue found by stress-testing this deliverable. `predict_scenario` also rejects NaN, infinite and non-numeric inputs with a clear error: without that check a NaN scenario returned an all-NaN matrix while reporting no range warning.

In [ ]:
def chain_multiply(matrices: list) -> np.ndarray:
    """Sequential matrix product P(t1) @ P(t2) @ ... @ P(tn)."""
    result = matrices[0]
    for m in matrices[1:]:
        result = result @ m
    return result


def validate_softmax_and_chain() -> dict:
    """
    Sanity checks the primitives against hand-verifiable cases before they
    are trusted on the real cascade / non-homogeneous chain.
    """
    checks = {}

    # 1. sigmoid(logit(p)) recovers p exactly, for a simple known probability.
    p = 0.3
    logit = np.log(p / (1 - p))
    recovered = 1.0 / (1.0 + np.exp(-logit))
    checks["sigmoid_inverts_logit"] = bool(np.isclose(recovered, p, atol=1e-12))

    # 2. A known, simple 2-state homogeneous chain: chain-multiplying n copies
    #    of P must equal P @ P @ ... (n times), cross-checked against
    #    np.linalg.matrix_power as an independent reference implementation.
    P = np.array([[0.9, 0.1], [0.2, 0.8]])
    n = 5
    via_chain_multiply = chain_multiply([P] * n)
    via_matrix_power = np.linalg.matrix_power(P, n)
    checks["chain_multiply_matches_matrix_power"] = bool(np.allclose(via_chain_multiply, via_matrix_power))

    # 3. Rows of the 2-state chain result must still sum to 1 (probability simplex preserved).
    checks["chain_result_rows_sum_to_one"] = bool(np.allclose(via_chain_multiply.sum(axis=1), 1.0))

    return checks


def forecast_transition_tensor(X: np.ndarray, beta_hat: dict) -> np.ndarray:
    """Same shape/semantics as markov_dgp.build_transition_tensor, but driven by beta_hat."""
    n = X.shape[0]
    tensor = np.zeros((n, config.N_TRANCHES, config.N_TRANCHES))
    for t in range(n):
        x_row = X[t]
        for tranche in ["IG", "NonIG", "Junk"]:
            idx = config.TRANCHES.index(tranche)
            tensor[t, idx, :] = cascade_row(tranche, x_row, beta_hat)
        tensor[t, config.DEFAULT_IDX, config.DEFAULT_IDX] = 1.0
    return tensor


def cumulative_sequence(matrix_tensor: np.ndarray) -> np.ndarray:
    """Returns shape (n, 4, 4): P_1, P_1@P_2, P_1@P_2@P_3, ..., i.e. P_hat_1..P_hat_n."""
    n = matrix_tensor.shape[0]
    out = np.zeros_like(matrix_tensor)
    running = np.eye(config.N_TRANCHES)
    for t in range(n):
        running = running @ matrix_tensor[t]
        out[t] = running
    return out


def evaluate_cumulative(p_true_cum: np.ndarray, p_forecast_cum: np.ndarray) -> dict:
    n = p_true_cum.shape[0]
    per_period = []
    for t in range(n):
        diff = p_true_cum[t] - p_forecast_cum[t]
        mse = float(np.mean(diff ** 2))
        frob = float(np.linalg.norm(diff, ord="fro"))
        per_period.append({"quarter_offset": t + 1, "mse": mse, "frobenius_norm": frob})

    overall_diff = p_true_cum - p_forecast_cum
    overall_mse = float(np.mean(overall_diff ** 2))
    overall_frob = float(np.linalg.norm(overall_diff.reshape(n, -1), ord="fro"))

    return {"per_period": per_period, "overall_mse": overall_mse, "overall_frobenius_norm": overall_frob}


def beta_dict_to_json(beta_dict: dict) -> dict:
    return {tr: {st: vec.tolist() for st, vec in stages.items()} for tr, stages in beta_dict.items()}


def compute_x_train_bounds(X_train: np.ndarray) -> dict:
    """
    Observed min/max of X_train's macro columns (skips X0, the intercept,
    which is always 1). Exported alongside beta_hat so any downstream
    consumer -- the Python forecast functions here, or the Sheets scenario
    calculator -- can tell whether a queried scenario is inside the range
    the model was actually fit on.
    """
    bounds = {}
    for i, name in enumerate(config.FEATURE_NAMES[1:], start=1):
        bounds[name] = [float(X_train[:, i].min()), float(X_train[:, i].max())]
    return bounds


def assess_scenario_range(x_row: np.ndarray, x_train_bounds: dict) -> list:
    """
    Returns a list of human-readable warnings (empty if the scenario is
    within the observed training range). Does not block the forecast --
    an extreme scenario may be a deliberate stress test -- just makes the
    extrapolation visible instead of silent.
    """
    warnings = []
    for i, name in enumerate(config.FEATURE_NAMES[1:], start=1):
        lo, hi = x_train_bounds[name]
        val = float(x_row[i])
        if val < lo or val > hi:
            warnings.append(f"{name}={val:.3f} is outside the observed training range [{lo:.3f}, {hi:.3f}] -- this forecast is an extrapolation")
    return warnings


def compute_extrapolation_cap_bounds(x_train_bounds: dict) -> dict:
    """
    Resolves config's per-feature cap settings into concrete [lo, hi] bounds
    for each of X1/X2/X3. X1 and X3 are multiples of their own observed
    training range; X2 is an explicit literal range (see config.py for why
    each was chosen that way). Exported to model_weights.json so Sheets/
    Excel only ever need to clip against three plain [lo, hi] pairs, with no
    need to replicate the "multiple vs. explicit" distinction downstream.
    """
    x1_lo, x1_hi = x_train_bounds["X1_delta_earnings"]
    x3_lo, x3_hi = x_train_bounds["X3_interaction"]
    return {
        "X1_delta_earnings": [x1_lo * config.EXTRAPOLATION_CAP_X1_MULTIPLE, x1_hi * config.EXTRAPOLATION_CAP_X1_MULTIPLE],
        "X2_delta_yield": list(config.EXTRAPOLATION_CAP_X2_BOUNDS),
        "X3_interaction": [x3_lo * config.EXTRAPOLATION_CAP_X3_MULTIPLE, x3_hi * config.EXTRAPOLATION_CAP_X3_MULTIPLE],
    }


def cap_scenario_to_bounds(x_row: np.ndarray, x_train_bounds: dict):
    """
    Clamps X1 (delta earnings) and X2 (delta yield) to their resolved cap
    bounds (see compute_extrapolation_cap_bounds), then recomputes
    X3 = X1*X2 from the ALREADY-capped X1/X2 and additionally clamps X3
    itself to its own resolved bounds. Capping the interaction term
    independently matters: capping X1 and X2 alone doesn't sufficiently
    bound their product, which can still reach far outside X3's own cap
    range even when X1 and X2 are each individually within theirs (verified
    numerically -- see README).

    Returns (x_row_capped, capped_dims: list[str]) so the caller can report
    exactly which inputs were adjusted, rather than silently substituting a
    different scenario than the one requested.
    """
    cap_bounds = compute_extrapolation_cap_bounds(x_train_bounds)
    x1_cap_lo, x1_cap_hi = cap_bounds["X1_delta_earnings"]
    x2_cap_lo, x2_cap_hi = cap_bounds["X2_delta_yield"]
    x3_cap_lo, x3_cap_hi = cap_bounds["X3_interaction"]

    x1_raw, x2_raw = float(x_row[1]), float(x_row[2])
    x1_capped = float(np.clip(x1_raw, x1_cap_lo, x1_cap_hi))
    x2_capped = float(np.clip(x2_raw, x2_cap_lo, x2_cap_hi))
    x3_raw = x1_capped * x2_capped
    x3_capped = float(np.clip(x3_raw, x3_cap_lo, x3_cap_hi))

    capped_dims = []
    if x1_capped != x1_raw:
        capped_dims.append(f"X1_delta_earnings: {x1_raw:.3f} -> {x1_capped:.3f} (cap [{x1_cap_lo:.3f}, {x1_cap_hi:.3f}])")
    if x2_capped != x2_raw:
        capped_dims.append(f"X2_delta_yield: {x2_raw:.3f} -> {x2_capped:.3f} (cap [{x2_cap_lo:.3f}, {x2_cap_hi:.3f}])")
    if x3_capped != x3_raw:
        capped_dims.append(f"X3_interaction: {x3_raw:.3f} -> {x3_capped:.3f} (cap [{x3_cap_lo:.3f}, {x3_cap_hi:.3f}])")

    return np.array([1.0, x1_capped, x2_capped, x3_capped]), capped_dims


def _assert_finite_scenario_input(value, label: str) -> None:
    """
    Scenario inputs must be real, finite numbers. Without this, NaN flows
    through np.clip and the sigmoid into an all-NaN matrix while the range
    check (NaN comparisons are always False) reports no warning, and +/-inf
    is silently clamped into a plausible-looking answer for an input that
    was never a real scenario.
    """
    is_number = isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool)
    if not is_number or not np.isfinite(value):
        raise ValueError(f"{label} must be a finite number in percentage points (got {value!r})")


def predict_scenario(delta_earnings: float, delta_yield: float, beta_hat: dict, x_train_bounds: dict) -> dict:
    """
    The exploratory "what if" entry point (what Sheets/Excel's scenario
    calculators call) -- distinct from forecast_transition_tensor, which is
    used for batch evaluation on real (always in-range) X_test data and
    stays uncapped/unmodified. Caps the input (see cap_scenario_to_bounds),
    builds the full 4x4 transition matrix from the capped values, and
    reports both the raw-vs-in-range check and what, if anything, was capped.
    Raises ValueError for NaN/inf/non-numeric inputs instead of returning garbage.
    """
    _assert_finite_scenario_input(delta_earnings, "delta_earnings")
    _assert_finite_scenario_input(delta_yield, "delta_yield")
    x_row_raw = np.array([1.0, delta_earnings, delta_yield, delta_earnings * delta_yield])
    range_warnings = assess_scenario_range(x_row_raw, x_train_bounds)
    x_row_capped, capped_dims = cap_scenario_to_bounds(x_row_raw, x_train_bounds)

    matrix = np.zeros((config.N_TRANCHES, config.N_TRANCHES))
    for tranche in ["IG", "NonIG", "Junk"]:
        idx = config.TRANCHES.index(tranche)
        matrix[idx, :] = cascade_row(tranche, x_row_capped, beta_hat)
    matrix[config.DEFAULT_IDX, config.DEFAULT_IDX] = 1.0

    return {
        "matrix": matrix,
        "range_warnings": range_warnings,
        "capped_dims": capped_dims,
        "was_capped": len(capped_dims) > 0,
    }


def export_model_weights(beta_hat: dict, path: str, x_train_bounds: dict) -> None:
    payload = {
        "tranches": config.TRANCHES,
        "default_idx": config.DEFAULT_IDX,
        "max_fall": config.MAX_FALL,
        "cascade_stages": config.CASCADE_STAGES,
        "upgrade_prob": config.UPGRADE_PROB,
        "feature_names": config.FEATURE_NAMES,
        "feature_units_note": "X1/X2 are delta_earnings/delta_yield in PERCENTAGE POINTS (1.0 = 100bp), X3 = X1*X2",
        "x_train_bounds": x_train_bounds,
        "x_train_bounds_note": "Observed min/max of X1/X2/X3 in the training data. Scenarios outside this range are extrapolations the model was never validated on.",
        "extrapolation_cap_bounds": compute_extrapolation_cap_bounds(x_train_bounds),
        "extrapolation_cap_note": "X1/X2/X3 are clamped to these [lo, hi] bounds before scenario prediction, to prevent sigmoid saturation (literal 0%/100%) on extreme inputs. Per-feature, not one shared multiplier -- see config.py for why each was chosen independently.",
        "beta_hat": beta_dict_to_json(beta_hat),
    }
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

In [ ]:
print('Softmax / chain validation:', validate_softmax_and_chain())

test_idx = feats['test_idx']
p_forecast_test = forecast_transition_tensor(feats['X_test'], beta_hat)
p_true_test = p_true[test_idx]

forecast_cum = cumulative_sequence(p_forecast_test)
true_cum = cumulative_sequence(p_true_test)
evaluation = evaluate_cumulative(true_cum, forecast_cum)
print('Overall MSE:', evaluation['overall_mse'])
print('Overall Frobenius norm:', evaluation['overall_frobenius_norm'])
print('1-quarter-ahead Frobenius norm:', evaluation['per_period'][0]['frobenius_norm'])
print(f"{config.N_TEST}-quarter-ahead Frobenius norm:", evaluation['per_period'][-1]['frobenius_norm'])

x_train_bounds = compute_x_train_bounds(feats['X_train'])
print()
print('=== Extrapolation range check on test quarters ===')
any_out_of_range = False
for t, x_row in enumerate(feats['X_test']):
    range_warnings = assess_scenario_range(x_row, x_train_bounds)
    if range_warnings:
        any_out_of_range = True
        print(f'test quarter {t+1}:', range_warnings)
if not any_out_of_range:
    print('all test quarters within the observed training range (expected, same generative process)')

print()
print('=== Resolved extrapolation cap bounds (per feature) ===')
print(compute_extrapolation_cap_bounds(x_train_bounds))

print()
print('=== predict_scenario demo: a severe combined stress scenario ===')
demo = predict_scenario(-2, 10, beta_hat, x_train_bounds)
print('IG row:', np.round(demo['matrix'][0], 4), ' capped:', demo['was_capped'])
for c in demo['capped_dims']:
    print('   ', c)

## Export `model_weights.json`

This is the file used by the Google Sheets and Excel deliverables (step 5). Now also exports the observed X_train bounds and the resolved per-feature extrapolation cap bounds, so both can flag out-of-range what-ifs and cap them the same way this notebook does. Running the cell below downloads it if you're in Colab, or just saves it to the working directory otherwise.

In [ ]:
export_model_weights(beta_hat, 'model_weights.json', x_train_bounds)
print('Wrote model_weights.json')

try:
    from google.colab import files
    files.download('model_weights.json')
except ImportError:
    pass  # not running in Colab -- file is saved locally, nothing more to do

## Step 5: Generate the Google Sheets engine (`Code.gs`)

The Sheets calculator runs on Apps Script (JavaScript), which cannot execute inside a Jupyter/Colab notebook, so this step *generates* the script instead of running it. The JavaScript engine is a fixed template of the same cascade-logit math used above. The fitted β̂, the training bounds and the extrapolation caps from `model_weights.json` are injected into it automatically, then read back out of the generated text and compared to the source, so the Sheet can never be pasted with stale or hand-copied weights.

Two safeguards are built in. The generator refuses weights the JavaScript engine cannot handle correctly (NaN, a missing stage, a renamed tranche, an IG upgrade probability above 0, inverted cap bounds). The generated engine rejects blank, text, NaN and infinite scenario inputs, and absurd horizons, with a clear error instead of returning a NaN matrix or hanging.

To use it: run the cells below, copy the printed script (or download `sheets/Code.gs` if you are in Colab), paste it into **Extensions → Apps Script**, then run `setupSheet()` once. Full walkthrough in `sheets/SETUP_GUIDE.md`.

In [ ]:
import json
import math
import os
import re

# The JS template hard-codes this 4-tranche structure (matMul4_, the
# IG/NonIG/Junk loop in buildTransitionMatrix_), so any other tranche layout
# must fail here rather than produce a Sheet that silently mis-indexes.
EXPECTED_TRANCHES = ["IG", "NonIG", "Junk", "Default"]
NON_DEFAULT_TRANCHES = ["IG", "NonIG", "Junk"]

# The only keys the JS engine reads. Anything else in model_weights.json
# (notes, future additions) is deliberately NOT embedded.
EMBEDDED_KEYS = [
    "tranches", "default_idx", "max_fall", "cascade_stages", "upgrade_prob",
    "feature_names", "x_train_bounds", "extrapolation_cap_bounds", "beta_hat",
]

_WEIGHTS_PLACEHOLDER = "__MODEL_WEIGHTS_JSON__"
_WEIGHTS_BLOCK_RE = re.compile(r"// BEGIN MODEL_WEIGHTS\nvar MODEL_WEIGHTS = (.*?);\n// END MODEL_WEIGHTS", re.S)

APPS_SCRIPT_TEMPLATE = r'''/**
 * Credit Migration Rate Sensitivity Model -- Google Sheets engine.
 *
 * GENERATED FILE -- do not edit by hand. Produced by src/sheets_export.py
 * (notebook Step 5 / run_pipeline.py) from model_weights.json, so the fitted
 * weights below are always the ones from the run that generated it. To change
 * the engine logic, edit the template in src/sheets_export.py; to refresh the
 * weights, re-run the pipeline/notebook and re-paste this whole file.
 *
 * This re-implements the Python cascade-logit Markov engine (src/markov_dgp.py
 * cascade_row + src/markov_engine.py forecast_transition_tensor) natively in
 * Apps Script, driven by the beta_hat exported from model_weights.json.
 *
 * IMPORTANT UNITS NOTE (matches src/matrix_builder.py): deltaEarnings and
 * deltaYield inputs are in PERCENTAGE POINTS, i.e. type 0.5 for +0.5% (50bp),
 * not 0.005. The model was fit on this scale -- feeding decimals in in a
 * different scale will silently produce near-zero / meaningless probabilities.
 *
 * EXTRAPOLATION CAPPING (matches src/markov_engine.predict_scenario): inputs
 * far outside the training range are clamped to extrapolation_cap_bounds
 * (per-feature [lo, hi] pairs, NOT one shared multiplier -- earnings and
 * yield saturate at very different multiples of their own training range,
 * see config.py) before the matrix math runs, so a wildly extreme scenario
 * can't saturate the sigmoid into a literal, historically unprecedented
 * 0%/100%. SCENARIO_WARNING() reports when this happens.
 *
 * SETUP: paste this whole file into Extensions > Apps Script, save, then run
 * setupSheet() once from the Apps Script editor (Run button) and approve the
 * one-time authorization prompt. See SETUP_GUIDE.md for the full walkthrough.
 */

// ---------------------------------------------------------------------------
// Model weights -- injected from model_weights.json by src/sheets_export.py.
// Everything below the END marker is fixed engine code.
// ---------------------------------------------------------------------------
// BEGIN MODEL_WEIGHTS
var MODEL_WEIGHTS = __MODEL_WEIGHTS_JSON__;
// END MODEL_WEIGHTS

// ---------------------------------------------------------------------------
// Core engine (mirrors src/markov_dgp.cascade_row exactly)
// ---------------------------------------------------------------------------

/**
 * Numerically stable logistic sigmoid, matching src/markov_dgp._sigmoid.
 * The naive 1/(1+exp(-z)) form can overflow for very negative z -- a real
 * risk once a scenario far outside the training range is queried (allowed
 * through with a warning, not blocked -- see checkScenarioRange_). Keeping
 * the exponent argument always <= 0 means it can only underflow toward 0
 * (safe), never overflow.
 */
function sigmoid_(z) {
  if (z >= 0) return 1.0 / (1.0 + Math.exp(-z));
  var expZ = Math.exp(z);
  return expZ / (1.0 + expZ);
}

function buildFeatureRow_(deltaEarnings, deltaYield) {
  var x1 = deltaEarnings;
  var x2 = deltaYield;
  return [1, x1, x2, x1 * x2];
}

function dot_(x, beta) {
  var s = 0;
  for (var i = 0; i < x.length; i++) s += x[i] * beta[i];
  return s;
}

// Longest horizon CUMULATIVE_STEADY will chain. Only exists so a typo like
// 1e9 fails with a clear error instead of hanging the sheet until Apps
// Script's custom-function timeout.
var MAX_HORIZON_QUARTERS = 1000;

/** Human-readable description of a bad argument, for error messages. */
function describeValue_(value) {
  if (value === "") return "an empty cell";
  if (typeof value === "string") return "text '" + value + "'";
  return String(value);
}

/**
 * Throws a clear error unless value is a finite number. Without this, a
 * blank cell / text / NaN flows through Math.max/Math.min and the sigmoid and
 * either produces an all-NaN matrix (with SCENARIO_WARNING reporting "OK") or
 * a cryptic "toFixed is not a function" -- both found by running this
 * engine under Node.
 */
function assertFiniteNumber_(value, label) {
  if (typeof value !== "number" || !isFinite(value)) {
    throw new Error(label + " must be a finite number (got " + describeValue_(value) +
      "). Enter it in percentage points, e.g. 0.5 for +0.5%.");
  }
}

function assertValidScenario_(deltaEarnings, deltaYield) {
  assertFiniteNumber_(deltaEarnings, "Delta Earnings");
  assertFiniteNumber_(deltaYield, "Delta Yield");
}

/** Throws a clear, Sheets-visible error for a misspelled/unknown tranche name. */
function assertValidTranche_(tranche) {
  if (MODEL_WEIGHTS.tranches.indexOf(tranche) === -1) {
    throw new Error("Unknown tranche '" + tranche + "'. Must be exactly one of: " + MODEL_WEIGHTS.tranches.join(", "));
  }
}

/**
 * Warns (does not block) when a scenario falls outside the observed
 * training range -- the model is linear-in-the-logit and was fit on macro
 * deltas roughly in the +/-1 percentage-point range; querying it far
 * outside that is an extrapolation the fit was never validated on. Mirrors
 * src/markov_engine.assess_scenario_range. Always evaluated against the RAW
 * (uncapped) input, so it still tells you honestly how extreme your typed
 * scenario was, even though the matrix math itself uses the capped version.
 * @return Array of warning strings, empty if the scenario is in range.
 */
function checkScenarioRange_(deltaEarnings, deltaYield) {
  assertValidScenario_(deltaEarnings, deltaYield);
  var xRow = buildFeatureRow_(deltaEarnings, deltaYield);
  var names = MODEL_WEIGHTS.feature_names.slice(1); // skip intercept
  var warnings = [];
  for (var i = 0; i < names.length; i++) {
    var bounds = MODEL_WEIGHTS.x_train_bounds[names[i]];
    var val = xRow[i + 1];
    if (val < bounds[0] || val > bounds[1]) {
      warnings.push(names[i] + "=" + val.toFixed(3) + " is outside training range [" +
        bounds[0].toFixed(3) + ", " + bounds[1].toFixed(3) + "] -- extrapolation");
    }
  }
  return warnings;
}

/**
 * Clamps X1/X2 to their resolved extrapolation_cap_bounds (per-feature
 * [lo, hi] pairs -- NOT one shared multiplier, see MODEL_WEIGHTS and
 * config.py for why each was chosen independently: earnings and the
 * interaction term are multiples of their own training range, yield is an
 * explicit literal range chosen from where the fitted model is actually
 * still informative). Recomputes X3 = X1*X2 from the ALREADY-capped X1/X2
 * and additionally clamps X3 itself to its own resolved bounds. Matches
 * src/markov_engine.cap_scenario_to_bounds exactly.
 *
 * Fixes a real issue found by stress-testing this deliverable: a wildly
 * out-of-range scenario pushed the sigmoid into full saturation, returning
 * a literal 0%/100% -- e.g. "100% of IG demotes in one quarter", which has
 * no historical precedent however "correct" it is for a linear model
 * extrapolated that far. Capping keeps severe-but-plausible stress
 * scenarios expressive while preventing that saturation.
 * @return {xRow: number[], cappedDims: string[]} -- cappedDims is empty if
 *   the scenario was already within the cap.
 */
function capScenarioToBounds_(deltaEarnings, deltaYield) {
  assertValidScenario_(deltaEarnings, deltaYield);
  var b1 = MODEL_WEIGHTS.extrapolation_cap_bounds.X1_delta_earnings;
  var b2 = MODEL_WEIGHTS.extrapolation_cap_bounds.X2_delta_yield;
  var b3 = MODEL_WEIGHTS.extrapolation_cap_bounds.X3_interaction;

  var x1 = Math.min(Math.max(deltaEarnings, b1[0]), b1[1]);
  var x2 = Math.min(Math.max(deltaYield, b2[0]), b2[1]);
  var x3Raw = x1 * x2;
  var x3 = Math.min(Math.max(x3Raw, b3[0]), b3[1]);

  var cappedDims = [];
  if (x1 !== deltaEarnings) cappedDims.push("Delta Earnings: " + deltaEarnings.toFixed(3) + " -> " + x1.toFixed(3) + " (cap [" + b1[0].toFixed(3) + ", " + b1[1].toFixed(3) + "])");
  if (x2 !== deltaYield) cappedDims.push("Delta Yield: " + deltaYield.toFixed(3) + " -> " + x2.toFixed(3) + " (cap [" + b2[0].toFixed(3) + ", " + b2[1].toFixed(3) + "])");
  if (x3 !== x3Raw) cappedDims.push("Interaction: " + x3Raw.toFixed(3) + " -> " + x3.toFixed(3) + " (cap [" + b3[0].toFixed(3) + ", " + b3[1].toFixed(3) + "])");

  return {xRow: [1, x1, x2, x3], cappedDims: cappedDims};
}

/** Decodes one tranche's cascading logit stages into a full 1x4 probability row. */
function cascadeRow_(tranche, xRow) {
  var idx = MODEL_WEIGHTS.tranches.indexOf(tranche);
  var u = MODEL_WEIGHTS.upgrade_prob[tranche];
  var maxFall = MODEL_WEIGHTS.max_fall[tranche];
  var betas = MODEL_WEIGHTS.beta_hat[tranche];

  var pFallGe1 = sigmoid_(dot_(xRow, betas.A));
  var pStay = 1.0 - pFallGe1;

  var pFallEq = {};
  if (maxFall === 1) {
    pFallEq[1] = pFallGe1;
  } else {
    var pFallGe2Given = sigmoid_(dot_(xRow, betas.B));
    pFallEq[1] = pFallGe1 * (1.0 - pFallGe2Given);
    var pFallGe2 = pFallGe1 * pFallGe2Given;
    if (maxFall === 2) {
      pFallEq[2] = pFallGe2;
    } else {
      var pEq3Given = sigmoid_(dot_(xRow, betas.C));
      pFallEq[2] = pFallGe2 * (1.0 - pEq3Given);
      pFallEq[3] = pFallGe2 * pEq3Given;
    }
  }

  var row = [0, 0, 0, 0];
  if (u > 0) row[idx - 1] = u;
  row[idx] += (1.0 - u) * pStay;
  for (var k in pFallEq) {
    row[idx + parseInt(k, 10)] += (1.0 - u) * pFallEq[k];
  }
  return row;
}

/** Full 4x4 transition matrix for a given quarter's macro scenario (capped -- see capScenarioToBounds_). */
function buildTransitionMatrix_(deltaEarnings, deltaYield) {
  var xRow = capScenarioToBounds_(deltaEarnings, deltaYield).xRow;
  var matrix = [];
  for (var i = 0; i < 4; i++) matrix.push([0, 0, 0, 0]);
  ["IG", "NonIG", "Junk"].forEach(function (tranche) {
    var idx = MODEL_WEIGHTS.tranches.indexOf(tranche);
    matrix[idx] = cascadeRow_(tranche, xRow);
  });
  matrix[MODEL_WEIGHTS.default_idx][MODEL_WEIGHTS.default_idx] = 1.0;
  return matrix;
}

function matMul4_(A, B) {
  var out = [];
  for (var i = 0; i < 4; i++) {
    var row = [];
    for (var j = 0; j < 4; j++) {
      var s = 0;
      for (var k = 0; k < 4; k++) s += A[i][k] * B[k][j];
      row.push(s);
    }
    out.push(row);
  }
  return out;
}

// ---------------------------------------------------------------------------
// Custom spreadsheet functions (usable directly as =PREDICT_ROW(...) etc.)
// ---------------------------------------------------------------------------

/**
 * Predicted transition row for one tranche under a given macro scenario.
 * @param {string} tranche One of "IG", "NonIG", "Junk", "Default".
 * @param {number} deltaEarnings Change in earnings growth, in percentage points (0.5 = +0.5%).
 * @param {number} deltaYield Change in 10Y yield, in percentage points (0.25 = +25bp).
 * @return A 1x4 array of probabilities [->IG, ->NonIG, ->Junk, ->Default].
 * @customfunction
 */
function PREDICT_ROW(tranche, deltaEarnings, deltaYield) {
  assertValidTranche_(tranche);
  assertValidScenario_(deltaEarnings, deltaYield);
  if (tranche === "Default") return [0, 0, 0, 1];
  var xRow = capScenarioToBounds_(deltaEarnings, deltaYield).xRow;
  return [cascadeRow_(tranche, xRow)];
}

/**
 * Full 4x4 predicted transition matrix for a given macro scenario (one quarter ahead).
 * @param {number} deltaEarnings Change in earnings growth, in percentage points.
 * @param {number} deltaYield Change in 10Y yield, in percentage points.
 * @return A 4x4 array, rows/cols ordered [IG, NonIG, Junk, Default].
 * @customfunction
 */
function PREDICT_MATRIX(deltaEarnings, deltaYield) {
  return buildTransitionMatrix_(deltaEarnings, deltaYield);
}

/**
 * N-quarter cumulative transition matrix, assuming the SAME macro scenario
 * persists every quarter for n quarters (a sustained-stress what-if). Uses
 * sequential chain multiplication, matching src/markov_engine.cumulative_sequence.
 * @param {number} deltaEarnings Change in earnings growth, in percentage points.
 * @param {number} deltaYield Change in 10Y yield, in percentage points.
 * @param {number} nQuarters Number of quarters to chain (whole number, 1 to 1000).
 * @return A 4x4 array, rows/cols ordered [IG, NonIG, Junk, Default].
 * @customfunction
 */
function CUMULATIVE_STEADY(deltaEarnings, deltaYield, nQuarters) {
  if (typeof nQuarters !== "number" || !isFinite(nQuarters) || Math.floor(nQuarters) !== nQuarters ||
      nQuarters < 1 || nQuarters > MAX_HORIZON_QUARTERS) {
    throw new Error("nQuarters must be a whole number between 1 and " + MAX_HORIZON_QUARTERS +
      " (got " + describeValue_(nQuarters) + ")");
  }
  var n = nQuarters;
  var single = buildTransitionMatrix_(deltaEarnings, deltaYield);
  var result = single;
  for (var q = 1; q < n; q++) {
    result = matMul4_(result, single);
  }
  return result;
}

/**
 * Human-readable check of whether a scenario is inside the range the model
 * was actually fit on, AND whether the input(s) got capped before the
 * matrix math ran (see capScenarioToBounds_). Not required for the matrix
 * math to run -- purely informational, put next to the scenario inputs so
 * an extreme what-if doesn't look as confident as an in-range one, and so
 * you know the displayed matrix reflects the CAPPED scenario, not your
 * literal typed value, whenever capping kicked in.
 * @param {number} deltaEarnings Change in earnings growth, in percentage points.
 * @param {number} deltaYield Change in 10Y yield, in percentage points.
 * @return A text warning, or "OK - within training range".
 * @customfunction
 */
function SCENARIO_WARNING(deltaEarnings, deltaYield) {
  var warnings = checkScenarioRange_(deltaEarnings, deltaYield);
  var capped = capScenarioToBounds_(deltaEarnings, deltaYield).cappedDims;
  if (warnings.length === 0) return "OK - within training range";
  var msg = "EXTRAPOLATION WARNING: " + warnings.join("; ");
  if (capped.length > 0) {
    msg += " | INPUT CAPPED before computing (result reflects the capped value, not your typed input): " + capped.join("; ");
  }
  return msg;
}

// ---------------------------------------------------------------------------
// One-click UI setup. Run this once from the Apps Script editor (Run button)
// after pasting this file in. Builds a "Scenario" sheet with input cells,
// both output grids, a range-check warning cell, and a heatmap
// conditional-formatting color scale.
// ---------------------------------------------------------------------------

function setupSheet() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var sheet = ss.getSheetByName("Scenario");
  if (sheet) ss.deleteSheet(sheet);
  sheet = ss.insertSheet("Scenario");

  sheet.getRange("A1").setValue("Credit Migration Scenario Calculator").setFontSize(16).setFontWeight("bold");

  // --- Inputs ---
  sheet.getRange("A3").setValue("Inputs (percentage points, e.g. 0.5 = +0.5%)").setFontWeight("bold");
  sheet.getRange("A4").setValue("Delta Earnings Growth");
  sheet.getRange("A5").setValue("Delta 10Y Yield");
  sheet.getRange("A6").setValue("Cumulative Horizon (quarters)");
  sheet.getRange("C4").setValue(0).setBackground("#fff3cd").setBorder(true, true, true, true, false, false);
  sheet.getRange("C5").setValue(0).setBackground("#fff3cd").setBorder(true, true, true, true, false, false);
  sheet.getRange("C6").setValue(4).setBackground("#fff3cd").setBorder(true, true, true, true, false, false);

  sheet.getRange("A7").setValue("Scenario check");
  sheet.getRange("C7").setFormula("=SCENARIO_WARNING($C$4,$C$5)").setFontStyle("italic");

  // --- Next-quarter matrix ---
  var headers = ["", "-> IG", "-> NonIG", "-> Junk", "-> Default"];
  var rowLabels = ["IG ->", "NonIG ->", "Junk ->", "Default ->"];

  sheet.getRange("A9").setValue("Next-Quarter Transition Matrix").setFontWeight("bold");
  sheet.getRange("A10:E10").setValues([headers]).setFontWeight("bold");
  sheet.getRange("A11:A14").setValues(rowLabels.map(function (r) { return [r]; })).setFontWeight("bold");
  sheet.getRange("B11").setFormula("=PREDICT_MATRIX($C$4,$C$5)");
  sheet.getRange("B11:E14").setNumberFormat("0.00%");

  // --- Cumulative matrix ---
  sheet.getRange("A17").setValue("N-Quarter Cumulative Transition Matrix (sustained scenario)").setFontWeight("bold");
  sheet.getRange("A18:E18").setValues([headers]).setFontWeight("bold");
  sheet.getRange("A19:A22").setValues(rowLabels.map(function (r) { return [r]; })).setFontWeight("bold");
  sheet.getRange("B19").setFormula("=CUMULATIVE_STEADY($C$4,$C$5,$C$6)");
  sheet.getRange("B19:E22").setNumberFormat("0.00%");

  // --- Heatmap conditional formatting on both grids ---
  [["B11:E14"], ["B19:E22"]].forEach(function (rangeA1) {
    var range = sheet.getRange(rangeA1[0]);
    var rule = SpreadsheetApp.newConditionalFormatRule()
      .setGradientMaxpointWithValue("#d73027", SpreadsheetApp.InterpolationType.NUMBER, "1")
      .setGradientMidpointWithValue("#ffffbf", SpreadsheetApp.InterpolationType.NUMBER, "0.5")
      .setGradientMinpointWithValue("#1a9850", SpreadsheetApp.InterpolationType.NUMBER, "0")
      .setRanges([range])
      .build();
    var rules = sheet.getConditionalFormatRules();
    rules.push(rule);
    sheet.setConditionalFormatRules(rules);
  });

  sheet.setColumnWidths(1, 5, 110);
  ss.setActiveSheet(sheet);
  SpreadsheetApp.flush();
}
'''


def _is_finite_number(v) -> bool:
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)


def _validate_weights(w: dict) -> None:
    """Raises ValueError for anything the JS template can't handle correctly."""
    missing = [k for k in EMBEDDED_KEYS if k not in w]
    if missing:
        raise ValueError(f"model weights are missing required keys: {missing}")
    if list(w["tranches"]) != EXPECTED_TRANCHES:
        raise ValueError(f"tranches must be exactly {EXPECTED_TRANCHES} (the JS engine hard-codes this layout), got {w['tranches']}")
    if w["default_idx"] != EXPECTED_TRANCHES.index("Default"):
        raise ValueError(f"default_idx must be {EXPECTED_TRANCHES.index('Default')}, got {w['default_idx']}")

    feature_names = list(w["feature_names"])
    if len(feature_names) != 4:
        raise ValueError(f"feature_names must have 4 entries (intercept + X1, X2, X3), got {feature_names}")
    macro_features = feature_names[1:]

    for block in ("x_train_bounds", "extrapolation_cap_bounds"):
        if sorted(w[block]) != sorted(macro_features):
            raise ValueError(f"{block} keys must be exactly {macro_features}, got {sorted(w[block])}")
        for name, pair in w[block].items():
            if len(pair) != 2 or not all(_is_finite_number(v) for v in pair) or pair[0] > pair[1]:
                raise ValueError(f"{block}[{name!r}] must be a finite [lo, hi] pair with lo <= hi, got {pair}")

    for tranche in NON_DEFAULT_TRANCHES:
        stages = list(w["cascade_stages"][tranche])
        if len(stages) != w["max_fall"][tranche]:
            raise ValueError(f"{tranche}: {len(stages)} cascade stages but max_fall={w['max_fall'][tranche]}")
        if sorted(w["beta_hat"][tranche]) != sorted(stages):
            raise ValueError(f"{tranche}: beta_hat stages {sorted(w['beta_hat'][tranche])} != cascade_stages {sorted(stages)}")
        for stage, vec in w["beta_hat"][tranche].items():
            if len(vec) != len(feature_names) or not all(_is_finite_number(v) for v in vec):
                raise ValueError(f"beta_hat[{tranche!r}][{stage!r}] must be {len(feature_names)} finite numbers, got {vec}")
        u = w["upgrade_prob"][tranche]
        if not (_is_finite_number(u) and 0 <= u < 1):
            raise ValueError(f"upgrade_prob[{tranche!r}] must be in [0, 1), got {u}")
    if w["upgrade_prob"]["IG"] != 0:
        raise ValueError("upgrade_prob['IG'] must be 0 -- nothing sits above IG, and the JS engine would write to row index -1")


def _embedded_subset(model_weights: dict) -> dict:
    return {k: model_weights[k] for k in EMBEDDED_KEYS}


def extract_embedded_weights(script: str) -> dict:
    """Reads the MODEL_WEIGHTS object back out of a generated Code.gs."""
    m = _WEIGHTS_BLOCK_RE.search(script)
    if not m:
        raise ValueError("no BEGIN/END MODEL_WEIGHTS block found in the script")
    return json.loads(m.group(1))


def verify_apps_script(script: str, model_weights: dict) -> bool:
    """Raises ValueError unless the weights embedded in `script` exactly equal `model_weights`."""
    expected = json.loads(json.dumps(_embedded_subset(model_weights), allow_nan=False))
    embedded = extract_embedded_weights(script)
    if embedded != expected:
        differing = [k for k in EMBEDDED_KEYS if embedded.get(k) != expected.get(k)]
        raise ValueError(f"embedded MODEL_WEIGHTS does not match the source weights (differs in: {differing})")
    return True


def _compact_json(obj, level: int = 0) -> str:
    """JSON with one dict key per line but number lists kept on one line (a beta vector reads as one row)."""
    if isinstance(obj, dict):
        pad = "  " * (level + 1)
        items = [f"{pad}{json.dumps(k)}: {_compact_json(v, level + 1)}" for k, v in obj.items()]
        return "{\n" + ",\n".join(items) + "\n" + "  " * level + "}"
    return json.dumps(obj, allow_nan=False)


def build_apps_script(model_weights: dict) -> str:
    """Returns the full Code.gs text with `model_weights` (the model_weights.json dict) injected."""
    _validate_weights(model_weights)
    weights_js = _compact_json(_embedded_subset(model_weights))
    script = APPS_SCRIPT_TEMPLATE.replace(_WEIGHTS_PLACEHOLDER, weights_js)
    verify_apps_script(script, model_weights)
    return script


def export_apps_script(weights_path: str, out_path: str) -> str:
    """Reads model_weights.json, writes the generated Code.gs to out_path, and returns its text."""
    with open(weights_path, encoding="utf-8") as f:
        model_weights = json.load(f)
    script = build_apps_script(model_weights)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8", newline="\n") as f:
        f.write(script)
    return script

In [ ]:
apps_script = export_apps_script('model_weights.json', 'sheets/Code.gs')
print(f"Wrote sheets/Code.gs ({len(apps_script.splitlines())} lines). Embedded weights verified identical to model_weights.json.")
print()
print('----- BEGIN Code.gs (copy everything between the markers) -----')
print(apps_script)
print('----- END Code.gs -----')

try:
    from google.colab import files
    files.download('sheets/Code.gs')
except ImportError:
    pass  # not running in Colab -- file is saved locally, nothing more to do

## Step 6: Generate the Excel scenario calculator (`.xlsx`)

Excel formulas can't be executed inside this notebook either, so -- like Step 5 -- this *generates* the workbook rather than running it: a fixed openpyxl template (README / Scenario / Model_Weights / a hidden Calc sheet holding the cascade-logit formulas and the MMULT cumulative-matrix chain), with the fitted β̂, training bounds and extrapolation caps from `model_weights.json` written in, then read back and compared to the source.

This replaced an earlier hand-built version that had two confirmed bugs (found by inspecting its raw XML, not guessed): its heatmap used Excel's unsupported-by-openpyxl `x14` extended conditional formatting anchored to a fixed absolute cell that pointed past the last real cumulative-matrix block and never tracked the selected horizon; and its displayed matrix cells were `TEXT()` strings, which conditional-formatting colour scales don't evaluate on. Both are fixed here: the displayed cells are plain numbers with `number_format` set directly, and the heatmaps are standard `ColorScaleRule`s applied straight to those same displayed ranges, so they're self-contained instead of pointing at a hand-computed row.

**What's verified and what isn't**: `verify_workbook` checks the embedded weights match `model_weights.json` and that the cumulative chain's own row bounds are internally consistent (exactly the class of bug above). There is no Excel/LibreOffice in this environment, so — unlike Step 5's JS, which is actually executed under Node — the formulas here are never run. The stress-test cell below instead replays a scenario's cached values from a real Excel recalculation of the previous workbook and confirms the Python engine still reproduces them.

To use it: run the cells below, then open `credit_migration_scenario_calculator.xlsx`. Close it first in Excel if you're regenerating an already-open copy (it'll otherwise fail to save with a permissions error).

In [ ]:
import json
import math
import os

from openpyxl import Workbook, load_workbook
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.formula import ArrayFormula

DEFAULT_N_QUARTERS = 60

HEADER_FONT = Font(bold=True)
TITLE_FONT = Font(bold=True, size=16)
SECTION_FONT = Font(bold=True)
INPUT_FILL = PatternFill("solid", fgColor="FFF3CD")
INPUT_BORDER = Border(*(Side(style="thin"),) * 4)
GREEN, YELLOW, RED = "1A9850", "FFFFBF", "D73027"  # same hex values as sheets/Code.gs's gradient, for a consistent look


def _col(i: int) -> str:
    return get_column_letter(i)


def _scenario_input_cells() -> dict:
    """
    Where Scenario's three input cells live. The ONE place this is decided --
    Calc's raw-X-vector formulas (`=Scenario!...`) and _write_scenario_sheet's
    own input-writing code both read it from here, instead of each
    independently assuming the same row numbers and risking silent
    divergence if the layout above them ever changes.
    """
    return {"earnings": "C4", "yield": "C5", "horizon": "C6"}


def _validate_weights(w: dict) -> None:
    """Raises ValueError for anything this template can't handle correctly. Deliberately the same rules as sheets_export._validate_weights -- both engines require the identical 4-tranche/3-2-1-cascade structure."""
    from sheets_export import _validate_weights as _shared_validate

    _shared_validate(w)


# ---------------------------------------------------------------------------
# Calc sheet: cascade-logit formula blocks, one per non-Default tranche.
# _cascade_rows returns (rows, final) where rows is [(label, {0: formula}), ...]
# (column offset 0 = column B) and final maps k=0 (stay), 1 (fall 1 notch),
# ... up to max_fall, to an index into `rows` -- used by the caller to write
# the tranche's summary row once real row numbers are known.
# ---------------------------------------------------------------------------

def _stable_sigmoid_formula(logit_cell: str) -> str:
    return f"IF({logit_cell}>=0,1/(1+EXP(-{logit_cell})),EXP({logit_cell})/(1+EXP({logit_cell})))"


def _cascade_rows(tranche: str, stages: list, beta_row_of: dict, x_range: str) -> tuple:
    max_fall = len(stages)
    if max_fall not in (1, 2, 3):
        raise ValueError(f"{tranche}: cascade with {max_fall} stages is not supported (only 1, 2 or 3 are)")

    rows = []  # (label, {0: formula_no_leading_eq})

    def add(label, formula):
        rows.append((label, {0: formula}))
        return len(rows) - 1  # index into `rows`, resolved to a real row number by the caller

    beta_A = beta_row_of[(tranche, stages[0])]
    i_logit_A = add("logit_A", f"SUMPRODUCT({x_range}, Model_Weights!C{beta_A}:F{beta_A})")
    i_p_ge1 = add("p(fall=1)" if max_fall == 1 else "p(fall>=1)", "__SIGMOID__" + str(i_logit_A))
    i_p_stay = add("p(stay)", "__1MINUS__" + str(i_p_ge1))
    final = {}

    if max_fall == 1:
        final[0] = i_p_stay
        final[1] = i_p_ge1
    else:
        beta_B = beta_row_of[(tranche, stages[1])]
        i_logit_B = add("logit_B", f"SUMPRODUCT({x_range}, Model_Weights!C{beta_B}:F{beta_B})")
        b_label = "p(fall=2 | fall>=1)" if max_fall == 2 else "p(fall>=2 | fall>=1)"
        i_p_B_given = add(b_label, "__SIGMOID__" + str(i_logit_B))
        i_p_eq1 = add("p(fall=1)", f"__MUL1MINUS__{i_p_ge1}__{i_p_B_given}")
        i_p_ge2 = add("p(fall=2)" if max_fall == 2 else "p(fall>=2)", f"__MUL__{i_p_ge1}__{i_p_B_given}")
        final[0] = i_p_stay
        final[1] = i_p_eq1

        if max_fall == 2:
            final[2] = i_p_ge2
        else:
            beta_C = beta_row_of[(tranche, stages[2])]
            i_logit_C = add("logit_C", f"SUMPRODUCT({x_range}, Model_Weights!C{beta_C}:F{beta_C})")
            i_p_C_given = add("p(fall=3 | fall>=2)", "__SIGMOID__" + str(i_logit_C))
            i_p_eq2 = add("p(fall=2)", f"__MUL1MINUS__{i_p_ge2}__{i_p_C_given}")
            i_p_eq3 = add("p(fall=3)", f"__MUL__{i_p_ge2}__{i_p_C_given}")
            final[2] = i_p_eq2
            final[3] = i_p_eq3

    return rows, final


def _resolve_placeholder(formula: str, row_addr: list) -> str:
    """row_addr[i] = 'B<row>' for the i-th row emitted by _cascade_rows -- turns the __SIGMOID__3 etc. placeholders into real formulas now that row numbers are known."""
    if formula.startswith("__SIGMOID__"):
        return "=" + _stable_sigmoid_formula(row_addr[int(formula[len("__SIGMOID__"):])])
    if formula.startswith("__1MINUS__"):
        return f"=1-{row_addr[int(formula[len('__1MINUS__'):])]}"
    if formula.startswith("__MUL1MINUS__"):
        a, b = formula[len("__MUL1MINUS__"):].split("__")
        return f"={row_addr[int(a)]}*(1-{row_addr[int(b)]})"
    if formula.startswith("__MUL__"):
        a, b = formula[len("__MUL__"):].split("__")
        return f"={row_addr[int(a)]}*{row_addr[int(b)]}"
    return "=" + formula


# ---------------------------------------------------------------------------
# Workbook assembly
# ---------------------------------------------------------------------------

def _write_model_weights_sheet(ws, w: dict, macro_features: list, tranches: list) -> dict:
    """Returns a dict of row lookups the Calc/Scenario sheets need: beta_row[(tranche,stage)], upgrade_row[tranche], bounds_row[feature], cap_row[feature]."""
    ws["A1"] = "Model weights (from model_weights.json)"
    ws["A1"].font = TITLE_FONT

    r = 3
    ws[f"A{r}"] = "beta_hat -- fitted macro-sensitivity coefficients per tranche/cascade stage"
    ws[f"A{r}"].font = SECTION_FONT
    r += 1
    for c, label in enumerate(["Tranche", "Stage", "Intercept (X0)", "Delta Earnings (X1)", "Delta Yield (X2)", "Interaction (X3)"]):
        cell = ws.cell(row=r, column=1 + c, value=label)
        cell.font = HEADER_FONT
    r += 1
    beta_row = {}
    for tranche in tranches:
        for stage in w["cascade_stages"][tranche]:
            ws.cell(row=r, column=1, value=tranche)
            ws.cell(row=r, column=2, value=stage)
            for c, v in enumerate(w["beta_hat"][tranche][stage]):
                ws.cell(row=r, column=3 + c, value=float(v))
            beta_row[(tranche, stage)] = r
            r += 1

    r += 1
    ws[f"A{r}"] = "Upgrade probabilities (fixed, non-macro-sensitive -- see README.md point 3)"
    ws[f"A{r}"].font = SECTION_FONT
    r += 1
    ws.cell(row=r, column=1, value="Tranche").font = HEADER_FONT
    ws.cell(row=r, column=2, value="Upgrade Prob").font = HEADER_FONT
    r += 1
    upgrade_row = {}
    for tranche in tranches:
        ws.cell(row=r, column=1, value=tranche)
        ws.cell(row=r, column=2, value=float(w["upgrade_prob"][tranche]))
        upgrade_row[tranche] = r
        r += 1

    def bounds_table(title, block_key, note=None):
        nonlocal r
        r += 1
        ws[f"A{r}"] = title
        ws[f"A{r}"].font = SECTION_FONT
        r += 1
        ws.cell(row=r, column=1, value="Feature").font = HEADER_FONT
        ws.cell(row=r, column=2, value="Min" if "CAP" not in title else "Cap Min").font = HEADER_FONT
        ws.cell(row=r, column=3, value="Max" if "CAP" not in title else "Cap Max").font = HEADER_FONT
        r += 1
        row_of = {}
        display = {"X1_delta_earnings": "Delta Earnings (X1)", "X2_delta_yield": "Delta Yield (X2)", "X3_interaction": "Interaction (X3)"}
        for feat in macro_features:
            lo, hi = w[block_key][feat]
            ws.cell(row=r, column=1, value=display.get(feat, feat))
            ws.cell(row=r, column=2, value=float(lo))
            ws.cell(row=r, column=3, value=float(hi))
            row_of[feat] = r
            r += 1
        if note:
            ws[f"A{r}"] = note
            r += 1
        return row_of

    bounds_row = bounds_table("Observed X_train bounds (extrapolation-warning reference)", "x_train_bounds")
    cap_row = bounds_table(
        "EXTRAPOLATION CAP BOUNDS (per feature -- output becomes static at/beyond these values)",
        "extrapolation_cap_bounds",
        note=("Note: per-feature, not one shared multiplier -- earnings and yield saturate at very different multiples "
              "of their own training range. The interaction (X3) bound applies to the recomputed X1(capped)*X2(capped) "
              "value, so a COMBINED scenario's plateau can begin before either X1 or X2 alone reaches its own bound above."))

    ws.column_dimensions["A"].width = 10
    ws.column_dimensions["B"].width = 8
    for col in "CDEF":
        ws.column_dimensions[col].width = 16
    return {"beta_row": beta_row, "upgrade_row": upgrade_row, "bounds_row": bounds_row, "cap_row": cap_row}


def _write_calc_sheet(ws, w: dict, lookup: dict, n_quarters: int, tranches: list) -> dict:
    beta_row, upgrade_row, cap_row = lookup["beta_row"], lookup["upgrade_row"], lookup["cap_row"]
    macro_feature_names = ["X1_delta_earnings", "X2_delta_yield", "X3_interaction"]

    ws["A1"] = "X vector: row 2 = CAPPED (drives all math below), row 3 = RAW (for the range-check warning only)"
    ws["A1"].font = HEADER_FONT
    for c, label in enumerate(["X0", "X1 (Earn)", "X2 (Yield)", "X3 (Inter.)"]):
        ws.cell(row=1, column=2 + c, value=label).font = HEADER_FONT

    x1cap, x2cap, x3cap = (cap_row[f] for f in macro_feature_names)
    ws["B2"] = 1
    ws["C2"] = f"=MIN(MAX(C3,Model_Weights!B{x1cap}),Model_Weights!C{x1cap})"
    ws["D2"] = f"=MIN(MAX(D3,Model_Weights!B{x2cap}),Model_Weights!C{x2cap})"
    ws["E2"] = f"=MIN(MAX(C2*D2,Model_Weights!B{x3cap}),Model_Weights!C{x3cap})"
    input_cells = _scenario_input_cells()
    ws["A3"] = "X vector (RAW, uncapped)"
    ws["B3"], ws["C3"], ws["D3"], ws["E3"] = 1, f"=Scenario!{input_cells['earnings']}", f"=Scenario!{input_cells['yield']}", "=C3*D3"
    ws["A4"] = "Capped? (text, blank if not)"
    ws["C4"] = '=IF(C2<>C3,"Delta Earnings: "&TEXT(C3,"0.000")&" -> "&TEXT(C2,"0.000"),"")'
    ws["D4"] = '=IF(D2<>D3,"Delta Yield: "&TEXT(D3,"0.000")&" -> "&TEXT(D2,"0.000"),"")'
    ws["E4"] = '=IF(E2<>E3,"Interaction: "&TEXT(E3,"0.000")&" -> "&TEXT(E2,"0.000"),"")'

    r = 7
    final_row_of_tranche = {}  # tranche -> row number of its "[->IG,...,->Default]" summary row
    for idx, tranche in enumerate(tranches):
        ws.cell(row=r, column=1, value=tranche).font = HEADER_FONT
        r += 1
        stages = w["cascade_stages"][tranche]
        rows, final = _cascade_rows(tranche, stages, beta_row, "$B$2:$E$2")
        row_addr = []  # row_addr[i] = "B<row>" for the i-th emitted row
        base = r
        for i, (label, colmap) in enumerate(rows):
            row_addr.append(f"B{base + i}")
        for i, (label, colmap) in enumerate(rows):
            ws.cell(row=base + i, column=1, value=label)
            ws.cell(row=base + i, column=2, value=_resolve_placeholder(colmap[0], row_addr))
        r = base + len(rows)

        u = float(w["upgrade_prob"][tranche])
        max_fall = len(stages)
        if u > 0:
            u_row = upgrade_row[tranche]
            ws.cell(row=r, column=1, value="u (upgrade prob)")
            ws.cell(row=r, column=2, value=f"=Model_Weights!B{u_row}")
            u_cell = f"B{r}"
            r += 1
        else:
            u_cell = None

        ws.cell(row=r, column=1, value=f"{tranche} row [{', '.join('->' + t for t in tranches + ['Default'])}]")
        # Column layout is absolute, by TARGET tranche index, not by how many columns precede the
        # tranche's own block: column 2+idx is always this tranche's own diagonal (stay), and any
        # upgrade -- one tier up -- always lands at column 2+(idx-1), wherever that falls. Previously
        # this used a hardcoded "the upgrade is always column B, data starts at column C", which is
        # only right for idx==1 (NonIG) and silently misplaced Junk's (idx==2) upgrade into the ->IG
        # column while leaving ->Default (column E) never written at all.
        diagonal_col = 2 + idx
        for c in range(2, 6):  # explicitly zero every column first so none is ever left unwritten
            ws.cell(row=r, column=c, value=0)
        if u_cell:
            ws.cell(row=r, column=diagonal_col - 1, value=f"={u_cell}")
        for k in range(0, max_fall + 1):
            src = row_addr[final[k]]
            ws.cell(row=r, column=diagonal_col + k, value=f"=(1-{u_cell})*{src}" if u_cell else f"={src}")
        final_row_of_tranche[tranche] = r
        r += 2

    # n = the transition matrix's actual dimension (every non-Default tranche, plus Default itself) --
    # driven by len(tranches), not a literal 4, so the M block and cumulative chain's row/column math
    # below scales to however many tranches model_weights actually declares.
    n = len(tranches) + 1
    last_col = _col(1 + n)

    m_start = r
    ws.cell(row=r - 1, column=1, value=f"M (Next-Quarter Transition Matrix) [{', '.join(tranches + ['Default'])}]").font = HEADER_FONT
    for idx, tranche in enumerate(tranches):
        src = final_row_of_tranche[tranche]
        for c in range(n):
            ws.cell(row=m_start + idx, column=2 + c, value=f"={_col(2 + c)}{src}")
    for c in range(n):  # Default row: absorbing, [0,...,0,1]
        ws.cell(row=m_start + n - 1, column=2 + c, value=1 if c == n - 1 else 0)

    chain_start = m_start + n + 1
    ws.cell(row=chain_start - 1, column=1,
            value=f"Cumulative transition matrix chain, quarters 1-{n_quarters} (each block = MMULT(previous block, M)). Block 1 = M itself.").font = HEADER_FONT
    m_range = f"$B${m_start}:${last_col}${m_start + n - 1}"
    prev_range = None
    for q in range(1, n_quarters + 1):
        block_row = chain_start + (q - 1) * n
        if q == 1:
            for r_off in range(n):  # block 1 is a plain copy of M -- every row, not just the top one
                for c in range(n):
                    ws.cell(row=block_row + r_off, column=2 + c, value=f"={_col(2 + c)}{m_start + r_off}")
        else:
            ref = f"{prev_range},{m_range}"
            cell = ws.cell(row=block_row, column=2, value=ArrayFormula(f"B{block_row}:{last_col}{block_row + n - 1}", f"=MMULT({ref})"))
        prev_range = f"$B${block_row}:${last_col}${block_row + n - 1}"
    last_row = chain_start + n_quarters * n - 1

    ws.column_dimensions["A"].width = 34
    return {"m_start": m_start, "chain_start": chain_start, "last_row": last_row, "bounds_row": lookup["bounds_row"], "n": n, "last_col": last_col}


def _write_scenario_sheet(ws, calc: dict, n_quarters: int, tranches: list) -> None:
    ws["A1"] = "Credit Migration Scenario Calculator"
    ws["A1"].font = TITLE_FONT

    input_cells = _scenario_input_cells()  # single source of truth, shared with Calc's raw-X-vector formulas
    r = 3
    ws.cell(row=r, column=1, value="Inputs (percentage points, e.g. 0.5 = +0.5%)").font = SECTION_FONT
    r += 1
    for text, default, cell_ref in (("Delta Earnings Growth", 0, input_cells["earnings"]),
                                     ("Delta 10Y Yield", 0, input_cells["yield"]),
                                     (f"Cumulative Horizon (quarters, 1-{n_quarters})", min(10, n_quarters), input_cells["horizon"])):
        if cell_ref != f"C{r}":
            raise ValueError(f"_scenario_input_cells()[...]={cell_ref!r} no longer matches this section's actual row ({r}) -- update it to match")
        ws.cell(row=r, column=1, value=text)
        cell = ws[cell_ref]
        cell.value = default
        cell.font = HEADER_FONT
        cell.fill = INPUT_FILL
        cell.border = INPUT_BORDER
        r += 1
    horizon_cell = input_cells["horizon"]

    b1, b2, b3 = (calc["bounds_row"][f] for f in ("X1_delta_earnings", "X2_delta_yield", "X3_interaction"))
    ws.cell(row=r, column=1, value="Scenario check")
    check_cell = ws["C" + str(r)]  # display column: same fixed column as the input cells (a layout choice, not data-dependent)
    check_cell.value = (
        f'=IF(OR(Calc!C3<Model_Weights!B{b1}, Calc!C3>Model_Weights!C{b1}, Calc!D3<Model_Weights!B{b2}, Calc!D3>Model_Weights!C{b2}, '
        f'Calc!E3<Model_Weights!B{b3}, Calc!E3>Model_Weights!C{b3}), "EXTRAPOLATION WARNING: input(s) outside observed training range" & '
        f'IF(OR(Calc!C4<>"",Calc!D4<>"",Calc!E4<>""), " | INPUT CAPPED (result reflects capped value): " & TRIM(Calc!C4&" "&Calc!D4&" "&Calc!E4), ""), '
        f'"OK - within training range")')
    check_cell.alignment = Alignment(wrap_text=True)
    r += 2

    targets = tranches + ["Default"]
    headers = [""] + [f"-> {t}" for t in targets]
    row_labels = [f"{t} ->" for t in targets]
    cf_rules = []  # (range, value_expr_for_that_range's_top-left-relative_cell)

    # Display cells use TEXT(...,"0.00%") rather than a numeric cell + number_format: confirmed by
    # directly inspecting the file after a real Excel/OneDrive AutoSave round-trip that the number
    # format (correct in the freshly generated file) gets silently stripped back to General by that
    # round-trip, on THIS specific class of formula-driven cell -- not something this environment can
    # prevent from the writing side. TEXT() bakes the "%" into the returned string itself, so nothing
    # downstream can strip it. That makes the displayed cells text, not numbers, so the heatmap can no
    # longer colour them by their own value (openpyxl ColorScaleRule needs numeric cells) -- instead,
    # each FormulaRule below reads the SAME hidden Calc-sheet numeric cell the display text was
    # derived from, computed via ROW()/COLUMN() rather than relying on Excel's implicit
    # relative-reference shifting across a CF range (the mechanism that broke the original hand-built
    # file for exactly this kind of block).
    def matrix_block(title, calc_range, row_index_of, row_index_of_cf):
        nonlocal r
        ws.cell(row=r, column=1, value=title).font = SECTION_FONT
        r += 1
        for c, h in enumerate(headers):
            if h:
                ws.cell(row=r, column=1 + c, value=h).font = HEADER_FONT
        r += 1
        first_data_row = r
        for i, rl in enumerate(row_labels):
            ws.cell(row=r, column=1, value=rl).font = HEADER_FONT
            for c in range(len(targets)):
                value_expr = f"INDEX({calc_range}, {row_index_of(i)}, {c + 1})"
                ws.cell(row=r, column=2 + c, value=f'=TEXT({value_expr},"0.00%")')
            r += 1

        cf_range = f"{_col(2)}{first_data_row}:{_col(1 + len(targets))}{first_data_row + len(targets) - 1}"
        cf_value_expr = f"INDEX({calc_range}, {row_index_of_cf(first_data_row)}, COLUMN()-1)"
        cf_rules.append((cf_range, cf_value_expr))
        r += 1

    m_range = f"Calc!$B${calc['m_start']}:${calc['last_col']}${calc['m_start'] + calc['n'] - 1}"
    matrix_block("Next-Quarter Transition Matrix", m_range,
                 row_index_of=lambda i: str(i + 1),
                 row_index_of_cf=lambda first_data_row: f"ROW()-{first_data_row}+1")

    horizon_abs = "$" + horizon_cell[0] + "$" + horizon_cell[1:]  # e.g. "C6" -> "$C$6"
    chain_range = f"Calc!$B${calc['chain_start']}:${calc['last_col']}${calc['last_row']}"

    def cum_display(i):  # wraps the whole TEXT() call, not just the INDEX -- "Invalid horizon" must stay a plain string
        return f"({horizon_abs}-1)*{calc['n']}+{i + 1}"

    def wrap_cum_cell_invalid_guard(formula_with_eq: str) -> str:
        inner = formula_with_eq[1:]  # strip the leading "="
        return f'=IF(OR({horizon_abs}<1,{horizon_abs}<>ROUND({horizon_abs},0),{horizon_abs}>{n_quarters}),"Invalid horizon",{inner})'

    matrix_block("N-Quarter Cumulative Transition Matrix (sustained scenario)", chain_range,
                 row_index_of=cum_display,
                 row_index_of_cf=lambda first_data_row: f"({horizon_abs}-1)*{calc['n']}+ROW()-{first_data_row}+1")

    # retrofit the invalid-horizon guard onto the cumulative block's display cells (matrix_block above
    # doesn't know which block is which, so this stays a small, explicit post-pass rather than another
    # branch parameter)
    cum_first_row = cf_rules[1][0].split(":")[0][1:]  # e.g. "B18:E21" -> "18"
    cum_first_row = int("".join(ch for ch in cum_first_row if ch.isdigit()))
    for rr in range(cum_first_row, cum_first_row + len(targets)):
        for cc in range(2, 2 + len(targets)):
            cell = ws.cell(row=rr, column=cc)
            cell.value = wrap_cum_cell_invalid_guard(cell.value)

    GREEN_FILL, YELLOW_FILL, RED_FILL = (PatternFill("solid", fgColor=c) for c in (GREEN, YELLOW, RED))
    for cf_range, value_expr in cf_rules:
        ws.conditional_formatting.add(cf_range, FormulaRule(formula=[f"{value_expr}>=0.66"], fill=RED_FILL))
        ws.conditional_formatting.add(cf_range, FormulaRule(formula=[f"AND({value_expr}>=0.33,{value_expr}<0.66)"], fill=YELLOW_FILL))
        ws.conditional_formatting.add(cf_range, FormulaRule(formula=[f"{value_expr}<0.33"], fill=GREEN_FILL))

    ws.cell(row=r, column=1, value="Note: output becomes static (flat) beyond these input bounds:")
    r += 1
    ws.cell(row=r, column=1, value="See the Model_Weights tab (EXTRAPOLATION CAP BOUNDS) for the exact current values and why.")

    dv = DataValidation(type="whole", formula1=1, formula2=n_quarters, showErrorMessage=True,
                         errorTitle="Invalid horizon", error=f"Enter a whole number of quarters from 1 to {n_quarters}.")
    ws.add_data_validation(dv)
    dv.add(horizon_cell)

    ws.column_dimensions["A"].width = 24
    ws.column_dimensions["B"].width = 12


def _write_readme_sheet(ws, w: dict, macro_features: list) -> None:
    x1c, x2c, x3c = (w["extrapolation_cap_bounds"][f] for f in macro_features)
    lines = [
        ("Credit Migration Scenario Calculator", True),
        ("", False),
        ("What this is", True),
        ("Pure-formula Excel port of the cascade-logit Markov engine from src/markov_dgp.py and src/markov_engine.py "
         "-- no macros, no add-ins, opens anywhere. Generated by src/excel_export.py from model_weights.json -- do not "
         "edit the Calc or Model_Weights tabs by hand, they will be overwritten on the next regeneration.", False),
        ("", False),
        ("How to use it", True),
        ("1. Go to the 'Scenario' tab.", False),
        ("2. Edit the two yellow input cells: Delta Earnings Growth and Delta 10Y Yield, both in PERCENTAGE POINTS "
         "(type 0.5 for +0.5%, i.e. +50bp -- not 0.005). This matches exactly how the Python model was fit.", False),
        (f"3. Edit Cumulative Horizon (quarters, 1-{DEFAULT_N_QUARTERS}) to choose how many quarters to compound the scenario over. "
         "If this is ever invalid (0, negative, non-integer), the Cumulative grid shows 'Invalid horizon' instead of an error.", False),
        ("4. The 'Scenario check' cell tells you if your inputs are within the range the model was actually fit on. "
         "If they're beyond the per-feature cap (see the Model_Weights tab), it also shows an INPUT CAPPED note -- "
         "inputs that extreme are clamped before the matrix math runs, so the sigmoid can't saturate into a literal, "
         'historically-unprecedented 0%/100% (e.g. "100% of IG demotes in one quarter"). The displayed matrix reflects '
         "the CAPPED value whenever that note appears, not your literal typed input.", False),
        ("5. The two 4x4 grids below show the next-quarter and N-quarter-cumulative transition matrices as real "
         "percentages, colour-scaled green (low probability) to red (high probability).", False),
        ("6. If a grid ever looks stuck (doesn't change when you edit the inputs), press Ctrl+Alt+F9 to force a full recalculation.", False),
        ("", False),
        ("IMPORTANT: where the output becomes static (flat)", True),
        (f"Delta Earnings Growth is clamped to [{x1c[0]:.3f}, {x1c[1]:.3f}]. Any input at or beyond that boundary computes "
         "the SAME result as the boundary value itself -- this is not a bug.", False),
        (f"Delta 10Y Yield is clamped to [{x2c[0]:.3f}, {x2c[1]:.3f}] -- an EXPLICIT range, not a multiplier of the training "
         "range. Yield's fitted sensitivity is smaller than earnings', so a small multiplier leaves it needlessly "
         "restricted while a large one overshoots into saturation -- this range was chosen from where the model is "
         "actually still informative, not derived mechanically.", False),
        (f"The interaction term (Delta Earnings x Delta Yield) is capped to [{x3c[0]:.3f}, {x3c[1]:.3f}] AFTER being "
         "recomputed from the already-capped values above -- capping the two factors alone doesn't sufficiently bound "
         "their product. This means a COMBINED scenario's plateau can begin before either input alone reaches its own boundary.", False),
        ("Past these boundaries the tool can no longer distinguish \"moderately extreme\" from \"wildly extreme\" -- that's "
         "the deliberate tradeoff of preventing saturation-driven nonsense, not a defect.", False),
        ("", False),
        ("Other tabs", True),
        ("Model_Weights -- the fitted beta_hat coefficients, observed training-data bounds, and the explicit per-feature "
         "capped-input boundaries, generated from model_weights.json. Reference only, not meant to be edited.", False),
        ("Calc (hidden) -- intermediate cascade-logit calculations and the pre-computed cumulative matrix chain. Unhide "
         "it (right-click any tab) if you want to see the full working.", False),
        ("", False),
        ("Source of truth", True),
        ("All beta_hat values here come directly from model_weights.json, generated by python run_pipeline.py. "
         "Re-run that (which now also regenerates this file) to refresh the numbers.", False),
    ]
    for i, (text, bold) in enumerate(lines, start=1):
        cell = ws.cell(row=i, column=1, value=text)
        cell.font = TITLE_FONT if i == 1 else (SECTION_FONT if bold else Font(bold=False))
    ws.column_dimensions["A"].width = 110


def build_workbook(model_weights: dict, n_quarters: int = DEFAULT_N_QUARTERS) -> Workbook:
    _validate_weights(model_weights)
    if not (isinstance(n_quarters, int) and n_quarters >= 1):
        raise ValueError(f"n_quarters must be a positive integer, got {n_quarters!r}")
    macro_features = list(model_weights["feature_names"][1:])
    tranches = list(model_weights["tranches"][:-1])  # every tranche but Default, in order -- validated above to be exactly ["IG","NonIG","Junk"]

    wb = Workbook()
    wb.remove(wb.active)
    ws_readme = wb.create_sheet("README")
    ws_scenario = wb.create_sheet("Scenario")
    ws_weights = wb.create_sheet("Model_Weights")
    ws_calc = wb.create_sheet("Calc")

    lookup = _write_model_weights_sheet(ws_weights, model_weights, macro_features, tranches)
    calc = _write_calc_sheet(ws_calc, model_weights, lookup, n_quarters, tranches)
    _write_scenario_sheet(ws_scenario, calc, n_quarters, tranches)
    _write_readme_sheet(ws_readme, model_weights, macro_features)

    ws_calc.sheet_state = "hidden"
    wb.active = wb.sheetnames.index("Scenario")
    from openpyxl.workbook.properties import CalcProperties
    wb.calculation = CalcProperties(fullCalcOnLoad=True)

    verify_workbook(wb, model_weights)
    return wb


def verify_workbook(wb_or_path, model_weights: dict) -> bool:
    """Raises ValueError unless the Model_Weights sheet of `wb_or_path` holds exactly `model_weights`'s numbers, and the Calc-sheet cumulative chain is structurally sound (no block referencing past the sheet's own data -- the exact class of bug this generator replaced)."""
    wb = load_workbook(wb_or_path, data_only=False) if isinstance(wb_or_path, str) else wb_or_path
    ws_weights, ws_calc = wb["Model_Weights"], wb["Calc"]
    macro_features = list(model_weights["feature_names"][1:])
    tranches = list(model_weights["tranches"][:-1])

    lookup = {
        "beta_row": {(t, s): r for t in tranches for s, r in
                     [(row[1], i) for i, row in enumerate(ws_weights.iter_rows(min_row=5, max_col=2, values_only=True), start=5) if row[0] == t]},
    }
    for tranche in tranches:
        for stage in model_weights["cascade_stages"][tranche]:
            r = lookup["beta_row"].get((tranche, stage))
            if r is None:
                raise ValueError(f"Model_Weights sheet is missing a row for {tranche}/{stage}")
            want = [float(v) for v in model_weights["beta_hat"][tranche][stage]]
            got = [ws_weights.cell(row=r, column=3 + c).value for c in range(len(want))]
            if any(g is None or abs(g - w) > 1e-12 for g, w in zip(got, want)):
                raise ValueError(f"Model_Weights beta row for {tranche}/{stage} = {got}, expected {want}")

    for r in range(1, ws_weights.max_row + 1):
        label = ws_weights.cell(row=r, column=1).value
        if label == "Tranche" and ws_weights.cell(row=r, column=2).value == "Upgrade Prob":
            for i, tranche in enumerate(tranches, start=r + 1):
                got_t, got_u = ws_weights.cell(row=i, column=1).value, ws_weights.cell(row=i, column=2).value
                want_u = float(model_weights["upgrade_prob"][got_t])
                if got_t != tranche or abs(got_u - want_u) > 1e-12:
                    raise ValueError(f"Model_Weights upgrade_prob row {i} = ({got_t}, {got_u}), expected ({tranche}, {want_u})")
            break

    for block_key in ("x_train_bounds", "extrapolation_cap_bounds"):
        header = "Min" if block_key == "x_train_bounds" else "Cap Min"
        for r in range(1, ws_weights.max_row + 1):
            if ws_weights.cell(row=r, column=2).value == header:
                for i, feat in enumerate(macro_features, start=r + 1):
                    lo, hi = float(ws_weights.cell(row=i, column=2).value), float(ws_weights.cell(row=i, column=3).value)
                    want_lo, want_hi = (float(v) for v in model_weights[block_key][feat])
                    if abs(lo - want_lo) > 1e-12 or abs(hi - want_hi) > 1e-12:
                        raise ValueError(f"Model_Weights {block_key} row for {feat} = [{lo},{hi}], expected [{want_lo},{want_hi}]")
                break

    _verify_cumulative_chain_bounds(wb)
    return True


def _verify_cumulative_chain_bounds(wb) -> None:
    """
    Regression test for exactly the historical bug (a heatmap rule pointed at
    Calc!B288, past the last real cumulative-matrix block, and never tracked
    the selected horizon): recovers the intended chain length from the
    Scenario sheet's own C6 data-validation bound (not a re-derived guess),
    locates the Calc sheet's cascade-block/chain markers by their labels, and
    checks every reference -- the chain's own internal MMULT formulas AND the
    Scenario sheet's cumulative-grid INDEX formula -- points at rows that
    actually exist and matches the chain's true, computed end row exactly.
    """
    import re

    ws_calc, ws_scenario = wb["Calc"], wb["Scenario"]
    horizon_cell = _scenario_input_cells()["horizon"]
    dv = next(iter(ws_scenario.data_validations.dataValidation), None)
    if dv is None or dv.sqref is None or horizon_cell not in str(dv.sqref):
        raise ValueError(f"Scenario!{horizon_cell} has no data validation -- can't recover the intended cumulative horizon to check against")
    n_quarters = int(dv.formula2)

    # Locate the two displayed matrix blocks by their own title labels, rather than assuming their
    # rows -- so this stays correct however many rows the input section above ends up using. Width
    # (n_targets = every tranche plus Default) comes from how many columns the next-quarter grid's
    # own header row actually filled in, not a literal 4 -- both the Calc-sheet block size below and
    # the completeness check reuse this SAME discovered value.
    next_q_title_row = cum_title_row = None
    for row in ws_scenario.iter_rows(min_col=1, max_col=1):
        for cell in row:
            if not isinstance(cell.value, str):
                continue
            if cell.value.startswith("Next-Quarter Transition Matrix"):
                next_q_title_row = cell.row
            elif cell.value.startswith("N-Quarter Cumulative Transition Matrix"):
                cum_title_row = cell.row
    if next_q_title_row is None or cum_title_row is None:
        raise ValueError("Scenario sheet is missing its 'Next-Quarter Transition Matrix' or 'N-Quarter Cumulative Transition Matrix' title label")
    next_q_first_row, cum_first_row = next_q_title_row + 2, cum_title_row + 2
    header_row = next_q_title_row + 1
    n_targets = sum(1 for c in range(2, ws_scenario.max_column + 1) if ws_scenario.cell(row=header_row, column=c).value)
    if n_targets == 0:
        raise ValueError("Next-Quarter Transition Matrix has no header columns to measure its width from")
    last_col = _col(1 + n_targets)

    label_row = {}
    for row in ws_calc.iter_rows(min_col=1, max_col=1):
        for cell in row:
            if isinstance(cell.value, str):
                if cell.value.startswith("M (Next-Quarter"):
                    label_row["m_header"] = cell.row
                elif cell.value.startswith("Cumulative transition matrix chain"):
                    label_row["chain_header"] = cell.row
    if "m_header" not in label_row or "chain_header" not in label_row:
        raise ValueError("Calc sheet is missing its 'M (Next-Quarter...' or 'Cumulative transition matrix chain...' label row")
    m_start = label_row["m_header"] + 1
    chain_start = label_row["chain_header"] + 1
    expected_last_row = chain_start + n_quarters * n_targets - 1

    # Regression test for a second, more basic bug caught by opening the file in real Excel: block 1
    # (the cumulative chain's first block, a plain copy of M) once only had its top row written -- the
    # other rows were silently blank, which every later MMULT then propagated forward as #VALUE! the
    # whole chain. Every cell in both the M block and block 1 must hold a real formula/value.
    for label, top_row in (("M block", m_start), ("cumulative block 1", chain_start)):
        for r_off in range(n_targets):
            for c_off in range(n_targets):
                cell = ws_calc.cell(row=top_row + r_off, column=2 + c_off)
                if cell.value is None:
                    raise ValueError(f"{label} is missing a value at {cell.coordinate} -- a downstream MMULT would silently propagate #VALUE! from here")

    array_refs = []
    for row in ws_calc.iter_rows():
        for cell in row:
            if isinstance(cell.value, ArrayFormula):
                array_refs.append((cell.row, cell.value.ref, cell.value.text))
    # n_quarters == 1 is a valid edge case (the horizon input allows it) with no MMULT at all -- block 1
    # is the whole chain, a plain copy of M, nothing to multiply. Only n_quarters > 1 requires blocks 2+.
    if not array_refs:
        if n_quarters > 1:
            raise ValueError(f"Calc sheet has no MMULT array formulas, but n_quarters={n_quarters} > 1 -- the chain beyond block 1 was not written")
    else:
        last_top_row, last_ref, _ = max(array_refs, key=lambda t: t[0])
        ref_end_row = int(re.search(r":\$?[A-Z]+\$?(\d+)", last_ref).group(1))
        if ref_end_row != expected_last_row:
            raise ValueError(f"last cumulative block's range ends at row {ref_end_row}, expected {expected_last_row} "
                              f"(n_quarters={n_quarters}, chain_start={chain_start})")

    for top_row, ref, text in array_refs:
        for m in re.finditer(r"\$?[A-Z]+\$(\d+)", text):
            referenced = int(m.group(1))
            if referenced > expected_last_row or referenced < m_start:
                raise ValueError(f"Calc!B{top_row}'s formula ({text}) references row {referenced}, outside [{m_start}, {expected_last_row}]")

    expected_calc_ref = f"Calc!$B${chain_start}:${last_col}${expected_last_row}"
    cum_cell = f"B{cum_first_row}"
    cum_formula = ws_scenario[cum_cell].value or ""
    if expected_calc_ref not in cum_formula:
        raise ValueError(f"Scenario!{cum_cell}'s cumulative-grid formula does not reference {expected_calc_ref} (found: {cum_formula!r})")

    # Display cells deliberately use TEXT(...,"0.00%") -- confirmed by inspecting a real file after an
    # Excel/OneDrive AutoSave round-trip that a plain numeric cell + number_format (the earlier design)
    # gets its format silently stripped back to General by that round-trip on this class of
    # formula-driven cell. TEXT() bakes the "%" into the returned string, immune to that.
    for coord in (f"B{next_q_first_row}", cum_cell):
        formula = ws_scenario[coord].value or ""
        if 'TEXT(' not in formula or '"0.00%"' not in formula:
            raise ValueError(f"Scenario!{coord} does not wrap its result in TEXT(...,\"0.00%\") -- required so the display survives an Excel/OneDrive AutoSave round-trip: {formula!r}")

    # Since the displayed cells are now text, the heatmap can't colour them by their own value -- it
    # must read the same hidden Calc-sheet numeric source the display text was derived from. Confirms
    # every colour tier (red/yellow/green) is present and its formula actually references an INDEX
    # into the SAME Calc range the display formula uses (calc_range/chain_range respectively) -- not a
    # hand-typed range that could silently drift from it, and specifically not the historical bug's
    # fixed absolute cell.
    next_q_calc_ref = f"Calc!$B${m_start}:${last_col}${m_start + n_targets - 1}"
    cum_calc_ref = f"Calc!$B${chain_start}:${last_col}${expected_last_row}"
    expected_cf = {
        f"B{next_q_first_row}:{last_col}{next_q_first_row + n_targets - 1}": next_q_calc_ref,
        f"B{cum_first_row}:{last_col}{cum_first_row + n_targets - 1}": cum_calc_ref,
    }
    got_ranges_by_sqref: dict = {}
    for cf in ws_scenario.conditional_formatting:
        got_ranges_by_sqref.setdefault(str(cf.sqref), []).extend(cf.rules)
    if set(got_ranges_by_sqref) != set(expected_cf):
        raise ValueError(f"Scenario sheet's conditional-formatting ranges are {set(got_ranges_by_sqref)}, expected {set(expected_cf)}")
    for rng, calc_ref in expected_cf.items():
        rules = got_ranges_by_sqref[rng]
        if len(rules) < 3:
            raise ValueError(f"Scenario!{rng} has only {len(rules)} conditional-format rule(s), expected 3 (red/yellow/green tiers)")
        formulas = " | ".join((r.formula[0] if r.formula else "") for r in rules)
        if "INDEX(" not in formulas or calc_ref not in formulas:
            raise ValueError(f"Scenario!{rng}'s conditional-format rules don't reference {calc_ref} (found: {formulas!r})")


def export_workbook(weights_path: str, out_path: str, n_quarters: int = DEFAULT_N_QUARTERS):
    with open(weights_path, encoding="utf-8") as f:
        model_weights = json.load(f)
    wb = build_workbook(model_weights, n_quarters)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    wb.save(out_path)
    return wb

In [ ]:
excel_out = 'credit_migration_scenario_calculator.xlsx'
try:
    wb = export_workbook('model_weights.json', excel_out, n_quarters=config.N_QUARTERS)
    print(f"Wrote {excel_out} (self-verified against model_weights.json)")
except PermissionError:
    print(f"SKIPPED: {excel_out} is open in Excel -- close it and re-run this cell")

try:
    from google.colab import files
    files.download(excel_out)
except ImportError:
    pass  # not running in Colab -- file is saved locally, nothing more to do

## Stress tests

The same automated suite used to stress-test `src/` locally (`tests/stress_suite.py`), run here literally inside this notebook's own Colab/Jupyter kernel -- not a local simulation of the notebook's code, the real thing, using every function already defined in the cells above.

Covers: boundary value analysis (extreme/huge/tiny inputs), negative testing (malformed input must fail loudly, never silently produce garbage), property-based fuzzing (invariants checked over 300 random scenarios spanning scale 1e-3 to 1e3), config-parameter extremes (e.g. zero smoothing epsilon should reintroduce `ln(0)`, proving it's load-bearing), the extrapolation-capping fix (extreme scenarios must trigger capping and never saturate to literal 0%/100%), malformed scenario input (NaN, infinity, text or None must raise a clear error, never return a NaN matrix), the Apps Script generator (must refuse weights the JavaScript engine can't handle and detect tampered embedded weights), the generated `Code.gs` itself, executed under Node and compared with the Python engine over 204 scenarios up to ±1000 plus malformed-input cases that used to return NaN or hang (a `[SKIP]` line means Node was not available in the kernel, so that part was **not** tested), the generated Excel workbook (weights in sync, cumulative-chain bounds internally consistent, a regression check against values a real copy of Excel actually calculated, and every cascade-logit formula in the Calc sheet symbolically evaluated and cross-checked against the Python engine across in-range AND outlier scenarios -- this last one is what actually caught a real bug: an earlier version hardcoded which column a tranche's upgrade probability landed in, which was only correct by coincidence for one tranche and silently misplaced Junk's; found by opening the file in real Excel, since the purely structural checks couldn't have caught it), determinism (same seed -> identical output), and cross-seed invariance of the pipeline's own internal sanity checks.

In [ ]:
def _safe(fn):
    try:
        return fn(), None
    except Exception as e:  # noqa: BLE001 -- intentionally broad, this is a test harness
        return None, f"{type(e).__name__}: {e}"


# ---------------------------------------------------------------------------
# 1. Extreme / out-of-range scenario stability (boundary value analysis)
# ---------------------------------------------------------------------------

def check_extreme_scenarios_stay_valid(ns) -> tuple:
    config = ns["config"]
    cascade_row = ns["cascade_row"]
    beta_hat = config.BETA_TRUE  # any valid beta dict works for this check

    extreme_rows = [
        np.array([1, 0, 0, 0]),
        np.array([1, 1e3, -1e3, -1e6]),
        np.array([1, -1e3, 1e3, -1e6]),
        np.array([1, 1e6, 1e6, 1e12]),
        np.array([1, -1e6, -1e6, 1e12]),
        np.array([1, 1e-10, 1e-10, 1e-20]),
    ]
    for x_row in extreme_rows:
        for tranche in ["IG", "NonIG", "Junk"]:
            result, err = _safe(lambda: cascade_row(tranche, x_row, beta_hat))
            if err:
                return False, f"cascade_row raised on extreme input {x_row} / {tranche}: {err}"
            if not np.all(np.isfinite(result)):
                return False, f"non-finite output for {x_row} / {tranche}: {result}"
            if np.any(result < -1e-9) or np.any(result > 1 + 1e-9):
                return False, f"probability out of [0,1] for {x_row} / {tranche}: {result}"
            if abs(sum(result) - 1.0) > 1e-6:
                return False, f"row does not sum to 1 for {x_row} / {tranche}: {result} (sum={sum(result)})"
    return True, f"{len(extreme_rows)} extreme scenarios x 3 tranches all stayed valid"


# ---------------------------------------------------------------------------
# 2. Negative testing: malformed inputs must fail loudly, not silently
# ---------------------------------------------------------------------------

def check_unknown_tranche_fails_loudly(ns) -> tuple:
    config = ns["config"]
    cascade_row = ns["cascade_row"]
    x_row = np.array([1, 0.1, 0.1, 0.01])
    result, err = _safe(lambda: cascade_row("NotATranche", x_row, config.BETA_TRUE))
    if err is None:
        return False, f"cascade_row('NotATranche', ...) did NOT raise -- silently returned {result}"
    return True, f"correctly raised: {err}"


def check_all_zero_weights_do_not_silently_corrupt_fit(ns) -> tuple:
    fit_ols = ns["fit_ols"]
    rng = np.random.default_rng(0)
    X = np.column_stack([np.ones(20), rng.normal(size=20), rng.normal(size=20), rng.normal(size=20)])
    Y = rng.normal(size=20)
    result, err = _safe(lambda: fit_ols(X, Y, weights=np.zeros(20)))
    if err is None and result is not None and np.all(np.isfinite(result["beta_hat"])):
        return False, f"all-zero weights produced a finite beta_hat with no warning/error: {result['beta_hat']}"
    return True, f"all-zero weights correctly failed or produced non-finite output ({err or 'non-finite result'})"


# ---------------------------------------------------------------------------
# 3. Config parameter extremes (monkeypatched, then restored)
# ---------------------------------------------------------------------------

def check_zero_smoothing_epsilon_breaks_as_expected(ns) -> tuple:
    """
    SMOOTHING_EPSILON exists specifically to stop ln(0). Setting it to 0
    should reintroduce that failure -- proving the epsilon is load-bearing,
    not decorative. If this DOESN'T fail, the smoothing logic has silently
    stopped depending on the epsilon somewhere.
    """
    config = ns["config"]
    stage_fn = ns["_stage_log_odds_and_weight"]
    original = config.SMOOTHING_EPSILON
    try:
        config.SMOOTHING_EPSILON = 0
        counts_row = np.array([50, 0, 0, 0])  # a zero count with zero smoothing -> ln(0)
        result, err = _safe(lambda: stage_fn(counts_row, 0, "A", 3))
        if err is None and np.isfinite(result[0]):
            return False, f"zero epsilon did NOT break ln(0) as expected -- got {result}, epsilon may be dead code"
        return True, f"zero epsilon correctly produced a non-finite/erroring result ({err or result})"
    finally:
        config.SMOOTHING_EPSILON = original


def check_tiny_company_pool_still_produces_valid_probabilities(ns) -> tuple:
    config = ns["config"]
    simulate_macro_paths = ns["simulate_macro_paths"]
    build_feature_matrix = ns["build_feature_matrix"]
    build_transition_tensor = ns["build_transition_tensor"]
    simulate_observed_counts = ns["simulate_observed_counts"]

    original = config.COMPANY_POOL_SIZE
    try:
        config.COMPANY_POOL_SIZE = 5  # extreme sparsity
        macro_df = simulate_macro_paths()
        X = build_feature_matrix(macro_df)
        p_true = build_transition_tensor(X, config.BETA_TRUE)
        result, err = _safe(lambda: simulate_observed_counts(p_true))
        if err:
            return False, f"pool_size=5 raised: {err}"
        counts = result
        if not np.all(counts >= 0):
            return False, "negative counts produced with tiny pool size"
        row_totals = counts.sum(axis=2)
        if not np.allclose(row_totals, config.COMPANY_POOL_SIZE):
            return False, f"row totals don't match pool size: {row_totals[0]}"
        return True, "pool_size=5 (extreme sparsity) still produced valid, non-negative counts summing correctly"
    finally:
        config.COMPANY_POOL_SIZE = original


# ---------------------------------------------------------------------------
# 4. Property-based fuzzing: invariants over many random inputs
# ---------------------------------------------------------------------------

def check_random_fuzzing_never_violates_invariants(ns, n_trials: int = 300) -> tuple:
    config = ns["config"]
    cascade_row = ns["cascade_row"]
    rng = np.random.default_rng(12345)

    violations = []
    for i in range(n_trials):
        scale = 10 ** rng.uniform(-3, 3)  # spans tiny to huge
        x1, x2 = rng.normal(scale=scale, size=2)
        x_row = np.array([1.0, x1, x2, x1 * x2])
        tranche = rng.choice(["IG", "NonIG", "Junk"])
        result, err = _safe(lambda: cascade_row(tranche, x_row, config.BETA_TRUE))
        if err:
            violations.append(f"trial {i} ({tranche}, x={x_row}): raised {err}")
            continue
        if not np.all(np.isfinite(result)):
            violations.append(f"trial {i} ({tranche}, x={x_row}): non-finite {result}")
        elif np.any(np.array(result) < -1e-9) or np.any(np.array(result) > 1 + 1e-9):
            violations.append(f"trial {i} ({tranche}, x={x_row}): out of [0,1] {result}")
        elif abs(sum(result) - 1.0) > 1e-6:
            violations.append(f"trial {i} ({tranche}, x={x_row}): sum={sum(result)}")

    if violations:
        return False, f"{len(violations)}/{n_trials} random trials violated an invariant. First: {violations[0]}"
    return True, f"{n_trials} random fuzzed scenarios (scale 1e-3 to 1e3) all satisfied every invariant"


def check_capping_prevents_saturation(ns) -> tuple:
    """
    Regression test for a real user-found issue: wildly out-of-range
    scenarios (e.g. a 10-percentage-point quarterly yield shock) pushed the
    sigmoid into full saturation, producing literal 0%/100% probabilities --
    "100% of IG demotes in one quarter" has no historical precedent, however
    mathematically "correct" it was for a linear-in-logit model extrapolated
    that far. predict_scenario now caps inputs to 3x the training bounds
    before the sigmoid math runs; this confirms extreme scenarios (a) are
    detected as needing capping and (b) no longer saturate to exactly 0 or 1.
    """
    config = ns["config"]
    predict_scenario = ns["predict_scenario"]
    simulate_macro_paths = ns["simulate_macro_paths"]
    build_feature_matrix = ns["build_feature_matrix"]
    compute_x_train_bounds = ns["compute_x_train_bounds"]

    macro_df = simulate_macro_paths()
    X = build_feature_matrix(macro_df)
    x_train_bounds = compute_x_train_bounds(X[: config.N_TRAIN])

    extreme_scenarios = [(-3, 15), (49, 4), (-5, -5), (1000, -1000)]
    for de, dy in extreme_scenarios:
        result, err = _safe(lambda: predict_scenario(de, dy, config.BETA_TRUE, x_train_bounds))
        if err:
            return False, f"predict_scenario raised on ({de},{dy}): {err}"
        if not result["was_capped"]:
            return False, f"scenario ({de},{dy}) should have triggered capping but didn't"
        ig_stay = result["matrix"][0, 0]
        if ig_stay >= 0.999999 or ig_stay <= 0.000001:
            return False, f"scenario ({de},{dy}): IG-stay={ig_stay} still saturated even after capping"
    return True, f"{len(extreme_scenarios)} extreme scenarios all triggered capping and avoided literal 0%/100% saturation"


def check_nonfinite_scenario_input_fails_loudly(ns) -> tuple:
    """
    Regression test for a silent failure found by probing predict_scenario:
    a NaN scenario returned an all-NaN matrix while reporting NO range
    warning (NaN comparisons are always False) and was_capped=True
    (nan != nan) -- contradictory flags around garbage output. +/-inf was
    silently clamped into a plausible-looking answer, and None/text gave a
    cryptic TypeError. All must now raise a clear error; valid input must
    still work.
    """
    config = ns["config"]
    predict_scenario = ns["predict_scenario"]
    simulate_macro_paths = ns["simulate_macro_paths"]
    build_feature_matrix = ns["build_feature_matrix"]
    compute_x_train_bounds = ns["compute_x_train_bounds"]

    X = build_feature_matrix(simulate_macro_paths())
    x_train_bounds = compute_x_train_bounds(X[: config.N_TRAIN])

    bad_inputs = [float("nan"), float("inf"), float("-inf"), None, "abc", "", "0.5", True]
    for bad in bad_inputs:
        for position, args in (("earnings", (bad, 0.1)), ("yield", (0.1, bad))):
            result, err = _safe(lambda: predict_scenario(*args, config.BETA_TRUE, x_train_bounds))
            if err is None:
                return False, f"predict_scenario did NOT raise for {position}={bad!r} -- silently returned a matrix (any NaN: {bool(np.isnan(result['matrix']).any())})"
            if "finite number" not in err:
                return False, f"{position}={bad!r} raised, but with an unhelpful message: {err}"
    ok, err = _safe(lambda: predict_scenario(0.1, -0.2, config.BETA_TRUE, x_train_bounds))
    if err:
        return False, f"a valid scenario raised after adding input validation: {err}"
    return True, f"{len(bad_inputs) * 2} malformed scenario inputs (NaN/inf/None/text/bool) all raised a clear error; valid input still works"


def _config_derived_weights(ns) -> dict:
    """
    A valid model_weights-shaped dict built from config.BETA_TRUE and the
    simulated macro path's training bounds. Any valid weights do for checking
    the generator/engine LOGIC; used where the real, fitted model_weights.json
    isn't available.
    """
    config = ns["config"]
    simulate_macro_paths = ns["simulate_macro_paths"]
    build_feature_matrix = ns["build_feature_matrix"]
    compute_x_train_bounds = ns["compute_x_train_bounds"]

    X = build_feature_matrix(simulate_macro_paths())
    bounds = compute_x_train_bounds(X[: config.N_TRAIN])
    return {
        "tranches": list(config.TRANCHES),
        "default_idx": config.DEFAULT_IDX,
        "max_fall": dict(config.MAX_FALL),
        "cascade_stages": {t: list(s) for t, s in config.CASCADE_STAGES.items()},
        "upgrade_prob": dict(config.UPGRADE_PROB),
        "feature_names": list(config.FEATURE_NAMES),
        "x_train_bounds": bounds,
        "extrapolation_cap_bounds": {name: [lo * 3, hi * 3] for name, (lo, hi) in bounds.items()},
        "beta_hat": {t: {s: [float(v) for v in vec] for s, vec in stages.items()} for t, stages in config.BETA_TRUE.items()},
    }


def _bad_weight_mutations(weights: dict) -> dict:
    """
    Invalid model_weights-shaped dicts, each broken in one specific way that
    a generator template (Sheets or Excel) is not written to handle safely:
    NaN beta, a missing cascade stage, a renamed tranche, an IG upgrade prob
    > 0 (both templates hard-code IG at index 0 with nothing above it),
    inverted or missing cap bounds. Shared between the Sheets and Excel
    generator-rejection checks since both call the same underlying
    sheets_export._validate_weights.
    """
    import copy

    def mutated(fn):
        w = copy.deepcopy(weights)
        fn(w)
        return w

    return {
        "NaN in a beta vector": mutated(lambda w: w["beta_hat"]["IG"]["A"].__setitem__(1, float("nan"))),
        "missing cascade stage": mutated(lambda w: w["beta_hat"]["IG"].pop("C")),
        "renamed tranche": mutated(lambda w: w.__setitem__("tranches", ["IG", "NonIG", "HY", "Default"])),
        "IG upgrade_prob > 0": mutated(lambda w: w["upgrade_prob"].__setitem__("IG", 0.1)),
        "cap bounds lo > hi": mutated(lambda w: w["extrapolation_cap_bounds"].__setitem__("X2_delta_yield", [10.0, -2.0])),
        "missing cap bounds": mutated(lambda w: w.pop("extrapolation_cap_bounds")),
    }


def check_apps_script_generator_rejects_bad_weights(ns) -> tuple:
    """
    The Sheets engine is generated from model_weights.json. The generator must
    refuse weights the JS engine can't handle correctly and must notice if the
    embedded weights were altered after generation -- otherwise a bad re-fit
    ships silently into the deployed Sheet.
    """
    build_apps_script = ns["build_apps_script"]
    verify_apps_script = ns["verify_apps_script"]
    weights = _config_derived_weights(ns)
    script, err = _safe(lambda: build_apps_script(weights))
    if err:
        return False, f"generator rejected a valid weights dict: {err}"

    bad_cases = _bad_weight_mutations(weights)
    for label, w in bad_cases.items():
        _, err = _safe(lambda: build_apps_script(w))
        if err is None:
            return False, f"generator accepted invalid weights: {label}"

    tampered = script.replace(str(weights["beta_hat"]["IG"]["A"][0]), "-1.0", 1)
    _, err = _safe(lambda: verify_apps_script(tampered, weights))
    if err is None:
        return False, "verify_apps_script accepted a script whose embedded weights were altered"
    return True, f"generator rejected {len(bad_cases)} kinds of invalid weights and verify caught a tampered script"


def check_excel_workbook_generator_rejects_bad_weights(ns) -> tuple:
    """Excel counterpart of check_apps_script_generator_rejects_bad_weights -- same underlying validator, same bad-weight cases."""
    build_workbook = ns["build_workbook"]
    weights = _config_derived_weights(ns)
    _, err = _safe(lambda: build_workbook(weights, 8))
    if err:
        return False, f"generator rejected a valid weights dict: {err}"

    bad_cases = _bad_weight_mutations(weights)
    for label, w in bad_cases.items():
        _, err = _safe(lambda: build_workbook(w, 8))
        if err is None:
            return False, f"generator accepted invalid weights: {label}"
    _, err = _safe(lambda: build_workbook(weights, 0))
    if err is None:
        return False, "generator accepted n_quarters=0"
    return True, f"generator rejected {len(bad_cases)} kinds of invalid weights, plus n_quarters=0"


def check_excel_workbook_file_is_in_sync_with_weights(ns) -> tuple:
    """
    The Excel workbook on disk must be buildable from, and verify against,
    the current model_weights.json -- catches a re-fit that forgot to
    regenerate the workbook (verify_workbook checks the embedded weights AND
    the cumulative-chain's own internal row bounds -- see its docstring for
    the historical bug this replaced).
    """
    import os

    verify_workbook = ns["verify_workbook"]
    weights_path, excel_path = ns.get("model_weights_path"), ns.get("excel_path")
    if not (weights_path and excel_path and os.path.exists(weights_path) and os.path.exists(excel_path)):
        return None, f"SKIPPED: model_weights.json and/or the Excel workbook not found ({weights_path}, {excel_path}) -- run the pipeline first"
    import json

    with open(weights_path, encoding="utf-8") as f:
        weights = json.load(f)
    try:
        verify_workbook(excel_path, weights)
    except PermissionError:
        return None, f"SKIPPED: {excel_path} is open in Excel -- can't read it to check (this is not evidence it's stale)"
    except ValueError as e:
        return False, f"STALE or broken: {e} -- regenerate with run_pipeline.py or notebook Step 6"
    return True, "the on-disk workbook's Model_Weights sheet and cumulative-chain bounds match the current model_weights.json"


def check_excel_workbook_matches_real_excel_calculation(ns) -> tuple:
    """
    Regression fixture: these next-quarter and cumulative matrices were
    CACHED VALUES read back from credit_migration_scenario_calculator.xlsx
    after it had genuinely been opened and recalculated by a real copy of
    Excel (captured 2026-09-29, scenario deltaEarnings=0, deltaYield=5, an
    out-of-training-range but uncapped yield shock) -- not something this
    environment computed. Confirms the Python engine still reproduces a
    result Excel's own formula engine actually produced.

    IMPORTANT LIMITATION: this checks the PYTHON engine against a frozen,
    real-Excel-verified number. It does NOT re-execute the CURRENT workbook's
    formulas (nothing in this environment can) -- that the newly generated
    file's formula shapes are structurally sound is covered separately by
    check_excel_workbook_file_is_in_sync_with_weights / verify_workbook.
    """
    predict_scenario = ns["predict_scenario"]

    # the fixture was captured from the REAL fitted model_weights.json (not config.BETA_TRUE),
    # so this check needs the real weights on disk to mean anything
    import json
    import os

    weights_path = ns.get("model_weights_path")
    if not (weights_path and os.path.exists(weights_path)):
        return None, f"SKIPPED: model_weights.json not found ({weights_path}) -- the golden fixture was fit from it, not from BETA_TRUE"
    with open(weights_path, encoding="utf-8") as f:
        w = json.load(f)
    beta_hat = {t: {s: np.array(v) for s, v in stages.items()} for t, stages in w["beta_hat"].items()}

    golden_next_q = np.array([
        [0.8485368304554919, 0.021594781852819173, 0.12916996395021146, 0.0006984237414774487],
        [0.04, 0.3858874585873844, 0.154539249009178, 0.4195732924034376],
        [0.0, 0.05, 0.014031143063444385, 0.9359688569365556],
        [0.0, 0.0, 0.0, 1.0],
    ])
    golden_q10 = np.array([
        [0.19816664004524817, 0.012728768170031407, 0.03292741994782935, 0.7561771718368908],
        [0.017373913916476517, 0.0012227774614950984, 0.002927530033603499, 0.9784757785884248],
        [0.001037120718690634, 8.031393910909449e-05, 0.00017754463834539336, 0.9987050207038549],
        [0.0, 0.0, 0.0, 1.0],
    ])
    result, err = _safe(lambda: predict_scenario(0, 5, beta_hat, w["x_train_bounds"]))
    if err:
        return False, f"predict_scenario(0, 5, ...) raised, but the same scenario computed fine in real Excel: {err}"
    diff_next_q = float(np.abs(result["matrix"] - golden_next_q).max())
    if diff_next_q > 1e-9:
        return False, f"next-quarter matrix differs from the real-Excel-cached values by {diff_next_q:.3e} (weights may have been re-fit since the fixture was captured)"

    running = np.eye(4)
    for _ in range(10):
        running = running @ result["matrix"]
    diff_q10 = float(np.abs(running - golden_q10).max())
    if diff_q10 > 1e-9:
        return False, f"10-quarter cumulative matrix differs from the real-Excel-cached values by {diff_q10:.3e}"
    return True, f"Python matches real-Excel-cached values for scenario (0, 5): next-Q diff={diff_next_q:.1e}, 10Q-cumulative diff={diff_q10:.1e}"


# ---------------------------------------------------------------------------
# Symbolic evaluator for the Calc sheet's cascade-logit formulas (NOT the
# MMULT cumulative chain, checked separately above). The generator only ever
# writes one of a handful of formula shapes, so this recomputes what each
# cell's formula actually encodes, top-to-bottom (every dependency points
# strictly upward), without needing Excel -- and lets a check cross-verify it
# against the already-trusted cascade_row for a given scenario.
#
# Why this exists: a hardcoded (rather than derived-from-tranche-position)
# column offset in an earlier version of excel_export.py silently misplaced
# Junk's upgrade probability into the wrong output column -- the row/column
# WAS populated, just with the wrong quantity, so the structural checks above
# (row bounds, no missing cells) could not have caught it. Found only by the
# user opening the file in real Excel. This evaluator would have; confirmed
# via check_excel_workbook_cascade_matches_python's own self-mutation test.
#
# Kept inline (not imported from a separate module) so the whole suite stays
# copy-pasteable into the notebook's stress-test cell, same as the embedded
# Node harness above.
#
# Raises on any formula shape it doesn't recognize, rather than silently
# skipping it, so a future template change not covered here fails loudly.
# ---------------------------------------------------------------------------
import re as _re

_CALC_FORMULA_PATTERNS = [
    (_re.compile(r"^=SUMPRODUCT\(\$B\$2:\$E\$2, Model_Weights!([A-Z]\d+):([A-Z]\d+)\)$"),
     lambda m, val, mw: sum(val[f"{c}2"] * mw(f"{chr(ord(m.group(1)[0]) + i)}{m.group(1)[1:]}") for i, c in enumerate("BCDE"))),
    (_re.compile(r"^=IF\(([A-Z]\d+)>=0,1/\(1\+EXP\(-\1\)\),EXP\(\1\)/\(1\+EXP\(\1\)\)\)$"),
     lambda m, val, mw: _stable_sigmoid_py(val[m.group(1)])),
    (_re.compile(r"^=1-([A-Z]\d+)$"), lambda m, val, mw: 1.0 - val[m.group(1)]),
    (_re.compile(r"^=\(1-([A-Z]\d+)\)\*\(1-([A-Z]\d+)\)$"), lambda m, val, mw: (1 - val[m.group(1)]) * (1 - val[m.group(2)])),
    (_re.compile(r"^=\(1-([A-Z]\d+)\)\*([A-Z]\d+)$"), lambda m, val, mw: (1 - val[m.group(1)]) * val[m.group(2)]),
    (_re.compile(r"^=([A-Z]\d+)\*\(1-([A-Z]\d+)\)$"), lambda m, val, mw: val[m.group(1)] * (1 - val[m.group(2)])),
    (_re.compile(r"^=([A-Z]\d+)\*([A-Z]\d+)$"), lambda m, val, mw: val[m.group(1)] * val[m.group(2)]),
    (_re.compile(r"^=Model_Weights!([A-Z]\d+)$"), lambda m, val, mw: mw(m.group(1))),
    (_re.compile(r"^=([A-Z]\d+)$"), lambda m, val, mw: val[m.group(1)]),
]


def _stable_sigmoid_py(z: float) -> float:
    import math
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)


def _evaluate_calc_sheet(ws_calc, ws_weights, x_row) -> dict:
    """{coordinate: float} for every recognized formula/literal cell in ws_calc, given x_row=[1,x1,x2,x3] seeded at B2:E2."""
    val = {"B2": float(x_row[0]), "C2": float(x_row[1]), "D2": float(x_row[2]), "E2": float(x_row[3])}

    def model_weights_val(ref: str) -> float:
        col = ord(ref[0]) - ord("A") + 1
        row = int(ref[1:])
        cell = ws_weights.cell(row=row, column=col).value
        if not isinstance(cell, (int, float)):
            raise ValueError(f"Model_Weights!{ref} is not numeric: {cell!r}")
        return float(cell)

    for row in ws_calc.iter_rows(min_row=5, min_col=2, max_col=5):  # rows 1-4 are X-vector setup, seeded above
        for cell in row:
            coord = cell.coordinate
            if coord in val:
                continue
            v = cell.value
            if v is None:
                continue
            if isinstance(v, (int, float)):
                val[coord] = float(v)
                continue
            if not isinstance(v, str) or not v.startswith("="):
                continue  # ArrayFormula (the MMULT chain) -- out of scope here
            for pattern, fn in _CALC_FORMULA_PATTERNS:
                m = pattern.match(v)
                if m:
                    val[coord] = fn(m, val, model_weights_val)
                    break
            else:
                raise ValueError(f"{coord}: unrecognized formula shape, evaluator needs updating: {v!r}")
    return val


def _find_tranche_final_rows(ws_calc) -> dict:
    """{tranche: row} for every "<tranche> row [->...]" summary row, located by label text (not a hardcoded row number)."""
    final_row = {}
    for row in ws_calc.iter_rows(min_col=1, max_col=1):
        for cell in row:
            if isinstance(cell.value, str) and cell.value.endswith("]") and " row [" in cell.value:
                final_row[cell.value.split(" row [")[0]] = cell.row
    return final_row


def check_excel_workbook_cascade_matches_python(ns) -> tuple:
    """
    Evaluates the Calc sheet's actual cascade-logit formulas (not just their
    presence/bounds) for a spread of scenarios -- in-range, and several
    extreme/outlier ones that get clamped by capping -- and compares every
    tranche's resulting row to cascade_row, the already-trusted Python
    engine. Also deliberately corrupts one tranche's output on a freshly
    built workbook and confirms this check actually notices (self-test run
    every time, not a one-off manual confirmation) -- see the module docstring
    above for why a purely structural check (rows populated, in bounds)
    cannot catch this class of bug.
    """
    import copy

    build_workbook = ns["build_workbook"]
    cascade_row = ns["cascade_row"]
    weights = _config_derived_weights(ns)
    wb, err = _safe(lambda: build_workbook(weights, 8))
    if err:
        return False, f"build_workbook raised on valid weights: {err}"
    ws_calc, ws_weights = wb["Calc"], wb["Model_Weights"]
    tranches = list(weights["tranches"][:-1])
    final_row = _find_tranche_final_rows(ws_calc)
    if set(final_row) != set(tranches):
        return False, f"could not locate a final row for every tranche: found {sorted(final_row)}, expected {sorted(tranches)}"

    beta_hat = {t: {s: np.array(v) for s, v in stages.items()} for t, stages in weights["beta_hat"].items()}
    cap = weights["extrapolation_cap_bounds"]

    def capped_x_row(de, dy):
        x1 = min(max(de, cap["X1_delta_earnings"][0]), cap["X1_delta_earnings"][1])
        x2 = min(max(dy, cap["X2_delta_yield"][0]), cap["X2_delta_yield"][1])
        x3 = min(max(x1 * x2, cap["X3_interaction"][0]), cap["X3_interaction"][1])
        return [1.0, x1, x2, x3]

    scenarios = [(0, 0), (0.1, -0.2), (-0.5, 0.3),
                 (-3, 10), (49, -1000), (1000, 1000), (-1000, -1000)]  # last four are deliberate outliers, beyond training range
    worst, worst_at = 0.0, None
    for de, dy in scenarios:
        x_row = capped_x_row(de, dy)
        result, err = _safe(lambda: _evaluate_calc_sheet(ws_calc, ws_weights, x_row))
        if err:
            return False, f"formula evaluator failed on scenario ({de},{dy}), x_row={x_row}: {err}"
        val = result
        for tranche in tranches:
            r = final_row[tranche]
            excel_row = np.array([val[f"{c}{r}"] for c in "BCDE"])
            py_row = cascade_row(tranche, np.array(x_row), beta_hat)
            diff = float(np.abs(excel_row - py_row).max())
            if diff > worst:
                worst, worst_at = diff, (de, dy, tranche)
    if worst > 1e-9:
        de, dy, tranche = worst_at
        return False, f"Calc formulas disagree with cascade_row by {worst:.3e} for {tranche} at scenario ({de},{dy})"

    # self-mutation: reproduce the ORIGINAL bug shape (upgrade probability shifted into the wrong
    # column, deepest-fall column left blank) on a copy, and confirm the comparison above would
    # have flagged it -- proves this check has teeth every run, not just when it was written.
    mutated_wb, err = _safe(lambda: build_workbook(weights, 8))
    if err:
        return False, f"build_workbook raised on a second valid build: {err}"
    m_calc = mutated_wb["Calc"]
    junk_row = _find_tranche_final_rows(m_calc).get("Junk")
    if junk_row is None or len(weights["cascade_stages"]["Junk"]) != 1 or weights["upgrade_prob"]["Junk"] <= 0:
        return False, "expected a single-stage Junk tranche with upgrade_prob > 0 to run the self-mutation check against"
    # correctly: B=0 (literal), C=upgrade, D=stay, E=fall-to-default -- shift everything one column
    # left (the original bug) so B=upgrade, C=stay, D=fall, and E is left blank/zero
    upgrade_f, stay_f, fall_f = (m_calc.cell(row=junk_row, column=c).value for c in (3, 4, 5))
    m_calc.cell(row=junk_row, column=2, value=upgrade_f)
    m_calc.cell(row=junk_row, column=3, value=stay_f)
    m_calc.cell(row=junk_row, column=4, value=fall_f)
    m_calc.cell(row=junk_row, column=5, value=0)
    mutated_val, err = _safe(lambda: _evaluate_calc_sheet(m_calc, mutated_wb["Model_Weights"], capped_x_row(-0.1, 0.2)))
    if err:
        return False, f"formula evaluator failed evaluating the self-mutated sheet: {err}"
    mutated_row = np.array([mutated_val[f"{c}{junk_row}"] for c in "BCDE"])
    py_row = cascade_row("Junk", np.array(capped_x_row(-0.1, 0.2)), beta_hat)
    mutation_diff = float(np.abs(mutated_row - py_row).max())
    if mutation_diff < 0.05:
        return False, f"self-mutation test failed to reproduce a detectable difference (diff={mutation_diff:.2e}) -- this check may not actually be exercising the bug class it claims to"

    return True, (f"{len(scenarios)} scenarios (incl. {sum(1 for de,dy in scenarios if abs(de)>3 or abs(dy)>3)} outliers beyond training range) "
                  f"x {len(tranches)} tranches: worst Calc-vs-Python diff = {worst:.2e}; self-mutation check confirmed this test catches the historical Junk-column bug (diff={mutation_diff:.2f})")


def check_default_row_always_absorbing(ns) -> tuple:
    config = ns["config"]
    build_transition_tensor = ns["build_transition_tensor"]
    build_feature_matrix = ns["build_feature_matrix"]
    simulate_macro_paths = ns["simulate_macro_paths"]

    macro_df = simulate_macro_paths()
    X = build_feature_matrix(macro_df)
    p_true = build_transition_tensor(X, config.BETA_TRUE)
    default_rows = p_true[:, config.DEFAULT_IDX, :]
    expected = np.zeros(config.N_TRANCHES)
    expected[config.DEFAULT_IDX] = 1.0
    if not np.allclose(default_rows, expected):
        return False, "Default row is not always exactly [0,0,0,1]"
    return True, "Default row is [0,0,0,1] in all 60 quarters"


# ---------------------------------------------------------------------------
# 5. Determinism / regression
# ---------------------------------------------------------------------------

def check_same_seed_gives_identical_output(ns) -> tuple:
    simulate_macro_paths = ns["simulate_macro_paths"]
    df1 = simulate_macro_paths(seed=999)
    df2 = simulate_macro_paths(seed=999)
    if not df1.equals(df2):
        return False, "same seed produced DIFFERENT macro paths across two runs -- non-determinism"
    return True, "same seed -> byte-identical macro path across two independent runs"


# ---------------------------------------------------------------------------
# 6. Cross-seed invariance of the pipeline's own sanity checks
# ---------------------------------------------------------------------------

def check_sanity_checks_hold_across_seeds(ns, n_seeds: int = 8) -> tuple:
    config = ns["config"]
    simulate_macro_paths = ns["simulate_macro_paths"]
    build_feature_matrix = ns["build_feature_matrix"]
    build_transition_tensor = ns["build_transition_tensor"]
    engine_sanity_check = ns["engine_sanity_check"]
    validate_softmax_and_chain = ns["validate_softmax_and_chain"]

    for seed in range(n_seeds):
        macro_df = simulate_macro_paths(seed=seed * 777 + 3)
        X = build_feature_matrix(macro_df)
        p_true = build_transition_tensor(X, config.BETA_TRUE)
        sanity = engine_sanity_check(X, p_true)
        for tranche, stages in sanity.items():
            for stage, r in stages.items():
                if r["max_abs_diff"] > 1e-6:
                    return False, f"seed={seed}: engine_sanity_check failed for {tranche}-{stage}: {r['max_abs_diff']}"
        checks = validate_softmax_and_chain()
        if not all(checks.values()):
            return False, f"seed={seed}: validate_softmax_and_chain failed: {checks}"
    return True, f"engine_sanity_check + validate_softmax_and_chain held across {n_seeds} different seeds"


# ---------------------------------------------------------------------------
# 7. Generated Google Sheets engine (Code.gs) executed under Node vs. Python
#
# Apps Script itself can't run here, but its custom functions are plain
# JavaScript, so the generated Code.gs is loaded into Node (SpreadsheetApp
# stubbed) and compared with the Python engine. If Node isn't installed these
# checks report SKIPPED -- which means "not tested", never "passed".
# ---------------------------------------------------------------------------

# Loads Code.gs into an isolated vm context, runs a list of calls read from
# stdin, prints one JSON result per call. A call that doesn't return within
# TIMEOUT_MS is killed and reported as an error, which is how a hang (e.g. an
# infinite loop) surfaces as a test failure instead of freezing the run.
NODE_HARNESS_JS = r"""
// Node harness for tests/js_parity_check.py.
//
// Loads sheets/Code.gs into an isolated vm context (Apps Script has no module
// system -- everything is a top-level global, which is exactly what vm gives
// us), stubs the one Apps Script global it references at load time, then runs
// a list of calls read from stdin and prints one JSON result per call.
//
// Usage: node js_parity_harness.js <path-to-Code.gs> < calls.json > results.json
// calls.json: [{"fn": "PREDICT_MATRIX", "args": [de, dy]}, ...]
// Each result is {"ok": <return value>} or {"error": "<message>"}; a call
// that doesn't return within TIMEOUT_MS is killed and reported as an error,
// which is how a hang (e.g. an infinite loop) surfaces as a test failure
// rather than freezing the test run.

const fs = require("fs");
const vm = require("vm");

const TIMEOUT_MS = 2000;
const codePath = process.argv[2];
const calls = JSON.parse(fs.readFileSync(0, "utf8"), (k, v) => (v === "__NaN__" ? NaN : v === "__Infinity__" ? Infinity : v === "__-Infinity__" ? -Infinity : v === "__undefined__" ? undefined : v));

const ctx = vm.createContext({ Math, JSON, parseInt, isFinite, SpreadsheetApp: {} });
vm.runInContext(fs.readFileSync(codePath, "utf8"), ctx, { filename: "Code.gs" });

const results = calls.map((call) => {
  ctx.__args = call.args;
  try {
    const value = vm.runInContext(`${call.fn}.apply(null, __args)`, ctx, { timeout: TIMEOUT_MS });
    return { ok: JSON.parse(JSON.stringify(value === undefined ? null : value)) };
  } catch (e) {
    return { error: String(e && e.message ? e.message : e) };
  }
});

process.stdout.write(JSON.stringify(results));
"""

PARITY_TOL = 1e-12
CHAIN_TOL = 1e-10
PARITY_EARNINGS = [-1000, -49, -5, -3, -2.5, -2, -0.78, -0.5, 0, 0.3, 0.65, 1, 1.94, 2, 5, 49, 1000]
PARITY_YIELD = [-1000, -5, -2, -0.37, 0, 0.31, 1, 5, 10, 15, 100, 1000]
PARITY_GRID = [(de, dy) for de in PARITY_EARNINGS for dy in PARITY_YIELD]
PARITY_HORIZONS = [1, 2, 4, 12, 60, 1000]
PARITY_HORIZON_SCENARIOS = [(0, 0), (-0.5, 0.3), (-3, 15), (49, 4), (1000, -1000)]


def _skip_without_node():
    import shutil

    if shutil.which("node") is None:
        return None, "SKIPPED: `node` not found on PATH, so the generated Code.gs was NOT executed (install Node.js to run this check)"
    return None


def _run_js(script: str, calls: list) -> list:
    """Runs `calls` ([{"fn": ..., "args": [...]}]) against the Code.gs text under Node; returns [{"ok": v} | {"error": msg}]."""
    import json
    import os
    import subprocess
    import tempfile

    with tempfile.TemporaryDirectory() as tmp:
        code_path = os.path.join(tmp, "Code.gs")
        harness_path = os.path.join(tmp, "harness.js")
        with open(code_path, "w", encoding="utf-8") as f:
            f.write(script)
        with open(harness_path, "w", encoding="utf-8") as f:
            f.write(NODE_HARNESS_JS)
        proc = subprocess.run(["node", harness_path, code_path], input=json.dumps(calls), capture_output=True,
                              text=True, encoding="utf-8", timeout=300)
    if proc.returncode != 0:
        raise RuntimeError(f"node harness failed:\n{proc.stderr}")
    return json.loads(proc.stdout)


def _parity_context(ns) -> dict:
    """The weights, and the Code.gs text generated from them, that the Node checks run against."""
    import json
    import os

    path = ns.get("model_weights_path")
    if path and os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            weights = json.load(f)
        source = "the fitted model_weights.json"
    else:
        weights = _config_derived_weights(ns)
        source = "BETA_TRUE-derived weights (model_weights.json not found)"
    beta_hat = {t: {s: np.array(v) for s, v in stages.items()} for t, stages in weights["beta_hat"].items()}
    return {"beta_hat": beta_hat, "bounds": weights["x_train_bounds"], "script": ns["build_apps_script"](weights), "source": source}


def check_sheets_engine_file_is_in_sync_with_weights(ns) -> tuple:
    """The Code.gs on disk must be exactly what the generator makes from the current model_weights.json."""
    import json
    import os

    weights_path, code_path = ns.get("model_weights_path"), ns.get("code_gs_path")
    if not (weights_path and code_path and os.path.exists(weights_path) and os.path.exists(code_path)):
        return None, f"SKIPPED: model_weights.json and/or Code.gs not found ({weights_path}, {code_path}) -- run the pipeline first"
    with open(weights_path, encoding="utf-8") as f:
        weights = json.load(f)
    with open(code_path, encoding="utf-8", newline="") as f:
        on_disk = f.read()
    if on_disk != ns["build_apps_script"](weights):
        try:
            ns["verify_apps_script"](on_disk, weights)
            return False, "embedded weights match model_weights.json, but the file differs from the generator output (hand-edited?) -- regenerate it"
        except ValueError as e:
            return False, f"STALE: {e} -- regenerate with run_pipeline.py or notebook Step 5"
    return True, f"Code.gs is byte-identical to the generator output for model_weights.json ({len(on_disk.splitlines())} lines)"


def check_sheets_engine_matrix_matches_python(ns) -> tuple:
    skipped = _skip_without_node()
    if skipped:
        return skipped
    ctx = _parity_context(ns)
    predict_scenario = ns["predict_scenario"]
    results = _run_js(ctx["script"], [{"fn": "PREDICT_MATRIX", "args": list(s)} for s in PARITY_GRID])
    worst, worst_at = 0.0, None
    for (de, dy), r in zip(PARITY_GRID, results):
        if "error" in r:
            return False, f"PREDICT_MATRIX({de}, {dy}) raised: {r['error']}"
        js = np.array(r["ok"], dtype=float)
        if (not np.all(np.isfinite(js)) or np.any(js < -1e-12) or np.any(js > 1 + 1e-12)
                or not np.allclose(js.sum(axis=1), 1.0, atol=1e-12)):
            return False, f"PREDICT_MATRIX({de}, {dy}) returned an invalid probability matrix: {r['ok']}"
        diff = float(np.abs(js - predict_scenario(de, dy, ctx["beta_hat"], ctx["bounds"])["matrix"]).max())
        if diff > worst:
            worst, worst_at = diff, (de, dy)
    if worst > PARITY_TOL:
        return False, f"JS vs Python max abs diff {worst:.3e} at {worst_at} exceeds {PARITY_TOL}"
    return True, f"{len(PARITY_GRID)} scenarios (in-range to +/-1000, {ctx['source']}): max abs diff JS vs Python = {worst:.2e}, every matrix valid"


def check_sheets_engine_row_and_cumulative_match_python(ns) -> tuple:
    skipped = _skip_without_node()
    if skipped:
        return skipped
    ctx = _parity_context(ns)
    predict_scenario = ns["predict_scenario"]

    tranches = ["IG", "NonIG", "Junk", "Default"]
    row_scenarios = [(0, 0), (-0.5, 0.3), (-3, 15), (49, 4)]
    row_results = iter(_run_js(ctx["script"], [{"fn": "PREDICT_ROW", "args": [t, de, dy]} for de, dy in row_scenarios for t in tranches]))
    for de, dy in row_scenarios:
        matrix = predict_scenario(de, dy, ctx["beta_hat"], ctx["bounds"])["matrix"]
        for i, t in enumerate(tranches):
            r = next(row_results)
            if "error" in r:
                return False, f"PREDICT_ROW({t}, {de}, {dy}) raised: {r['error']}"
            row = r["ok"][0] if isinstance(r["ok"][0], list) else r["ok"]
            if np.abs(np.array(row) - matrix[i]).max() > PARITY_TOL:
                return False, f"PREDICT_ROW({t}, {de}, {dy}) differs from the Python matrix row"

    calls = [{"fn": "CUMULATIVE_STEADY", "args": [de, dy, n]} for de, dy in PARITY_HORIZON_SCENARIOS for n in PARITY_HORIZONS]
    cum_results = iter(_run_js(ctx["script"], calls))
    worst = 0.0
    for de, dy in PARITY_HORIZON_SCENARIOS:
        single = predict_scenario(de, dy, ctx["beta_hat"], ctx["bounds"])["matrix"]
        running, chain = np.eye(4), {}
        for q in range(1, max(PARITY_HORIZONS) + 1):
            running = running @ single  # sequential product, as in cumulative_sequence
            if q in PARITY_HORIZONS:
                chain[q] = running.copy()
        for n in PARITY_HORIZONS:
            r = next(cum_results)
            if "error" in r:
                return False, f"CUMULATIVE_STEADY({de}, {dy}, {n}) raised: {r['error']}"
            js = np.array(r["ok"], dtype=float)
            if not np.all(np.isfinite(js)) or not np.allclose(js.sum(axis=1), 1.0, atol=1e-9):
                return False, f"CUMULATIVE_STEADY({de}, {dy}, {n}) is not a valid stochastic matrix"
            worst = max(worst, float(np.abs(js - chain[n]).max()))
    if worst > CHAIN_TOL:
        return False, f"cumulative JS vs Python max abs diff {worst:.3e} exceeds {CHAIN_TOL}"
    return True, f"{len(row_scenarios) * len(tranches)} tranche rows and {len(calls)} cumulative matrices (n up to {max(PARITY_HORIZONS)}) match Python; worst cumulative diff {worst:.2e}"


def check_sheets_engine_warning_flags_match_python(ns) -> tuple:
    skipped = _skip_without_node()
    if skipped:
        return skipped
    ctx = _parity_context(ns)
    predict_scenario = ns["predict_scenario"]
    results = _run_js(ctx["script"], [{"fn": "SCENARIO_WARNING", "args": list(s)} for s in PARITY_GRID])
    for (de, dy), r in zip(PARITY_GRID, results):
        if "error" in r:
            return False, f"SCENARIO_WARNING({de}, {dy}) raised: {r['error']}"
        py = predict_scenario(de, dy, ctx["beta_hat"], ctx["bounds"])
        msg = r["ok"]
        if bool(py["range_warnings"]) != msg.startswith("EXTRAPOLATION WARNING"):
            return False, f"({de}, {dy}): Python out-of-range={bool(py['range_warnings'])} but JS said {msg!r}"
        if py["was_capped"] != ("INPUT CAPPED" in msg):
            return False, f"({de}, {dy}): Python was_capped={py['was_capped']} but JS said {msg!r}"
    return True, f"range-warning and INPUT CAPPED flags agree with Python on all {len(PARITY_GRID)} scenarios"


def check_sheets_engine_rejects_malformed_input(ns) -> tuple:
    """
    Regression test for three bugs found by running the JavaScript under Node:
    a NaN input returned an all-NaN matrix while SCENARIO_WARNING said "OK";
    a blank/text input threw the cryptic "toFixed is not a function"; and a
    horizon of Infinity or 1e9 hung until the timeout (2.7 was also silently
    floored). All must now raise a clear, specific error.
    """
    skipped = _skip_without_node()
    if skipped:
        return skipped
    ctx = _parity_context(ns)
    nan, inf = "__NaN__", "__Infinity__"  # the harness turns these markers into real NaN/Infinity
    finite = "must be a finite number"
    horizon = "whole number between 1 and"
    cases = [
        ("blank scenario cell", "PREDICT_MATRIX", ["", 0.1], finite),
        ("text scenario", "PREDICT_MATRIX", [0.1, "abc"], finite),
        ("NaN scenario", "PREDICT_MATRIX", [nan, 0.1], finite),
        ("Infinity scenario", "PREDICT_MATRIX", [0.1, inf], finite),
        ("missing argument", "PREDICT_MATRIX", [0.1, "__undefined__"], finite),
        ("NaN into PREDICT_ROW", "PREDICT_ROW", ["IG", nan, 0.1], finite),
        ("NaN into the Default row", "PREDICT_ROW", ["Default", nan, 0.1], finite),
        ("NaN into SCENARIO_WARNING (must not say OK)", "SCENARIO_WARNING", [nan, 0.1], finite),
        ("blank into SCENARIO_WARNING", "SCENARIO_WARNING", ["", 0.1], finite),
        ("horizon = Infinity (used to hang)", "CUMULATIVE_STEADY", [0, 0, inf], horizon),
        ("horizon = 1e9 (used to hang)", "CUMULATIVE_STEADY", [0, 0, 1e9], horizon),
        ("horizon as text '4'", "CUMULATIVE_STEADY", [0, 0, "4"], horizon),
        ("horizon = 2.7 (used to floor silently)", "CUMULATIVE_STEADY", [0, 0, 2.7], horizon),
        ("horizon = 0", "CUMULATIVE_STEADY", [0, 0, 0], horizon),
        ("horizon = -1", "CUMULATIVE_STEADY", [0, 0, -1], horizon),
        ("horizon = NaN", "CUMULATIVE_STEADY", [0, 0, nan], horizon),
        ("horizon blank", "CUMULATIVE_STEADY", [0, 0, ""], horizon),
        ("horizon 1001 (over the limit)", "CUMULATIVE_STEADY", [0, 0, 1001], horizon),
        ("unknown tranche", "PREDICT_ROW", ["Bogus", 0, 0], "Unknown tranche"),
    ]
    results = _run_js(ctx["script"], [{"fn": fn, "args": args} for _, fn, args, _ in cases])
    for (label, _, _, expected), r in zip(cases, results):
        if "error" not in r:
            return False, f"{label}: did NOT raise -- silently returned {str(r['ok'])[:80]}"
        if expected not in r["error"]:
            return False, f"{label}: raised, but message {r['error']!r} lacks {expected!r}"
    return True, f"all {len(cases)} malformed-input cases raised a clear, specific error (no silent NaN, no hang)"


ALL_CHECKS = [
    check_extreme_scenarios_stay_valid,
    check_unknown_tranche_fails_loudly,
    check_all_zero_weights_do_not_silently_corrupt_fit,
    check_zero_smoothing_epsilon_breaks_as_expected,
    check_tiny_company_pool_still_produces_valid_probabilities,
    check_random_fuzzing_never_violates_invariants,
    check_capping_prevents_saturation,
    check_nonfinite_scenario_input_fails_loudly,
    check_apps_script_generator_rejects_bad_weights,
    check_excel_workbook_generator_rejects_bad_weights,
    check_default_row_always_absorbing,
    check_same_seed_gives_identical_output,
    check_sanity_checks_hold_across_seeds,
    check_sheets_engine_file_is_in_sync_with_weights,
    check_sheets_engine_matrix_matches_python,
    check_sheets_engine_row_and_cumulative_match_python,
    check_sheets_engine_warning_flags_match_python,
    check_sheets_engine_rejects_malformed_input,
    check_excel_workbook_file_is_in_sync_with_weights,
    check_excel_workbook_matches_real_excel_calculation,
    check_excel_workbook_cascade_matches_python,
]


def run_all(ns, label: str, checks=None) -> bool:
    """Runs `checks` (default: ALL_CHECKS). A check returning (None, msg) is SKIPPED: not a failure, but reported loudly as untested."""
    checks = ALL_CHECKS if checks is None else checks
    print(f"\n{'=' * 70}\nSTRESS SUITE: {label}\n{'=' * 70}")
    all_passed = True
    n_skipped = 0
    for check in checks:
        try:
            passed, message = check(ns)
        except Exception as e:  # a check itself blowing up is also a FAIL, not a crash
            passed, message = False, f"check raised unexpectedly: {type(e).__name__}: {e}"
        if passed is None:
            status = "SKIP"
            n_skipped += 1
        else:
            status = "PASS" if passed else "FAIL"
            if not passed:
                all_passed = False
        print(f"  [{status}] {check.__name__}: {message}")
    summary = "ALL PASSED" if all_passed else "AT LEAST ONE FAILURE"
    if all_passed and n_skipped:
        summary += f" ({n_skipped} check(s) SKIPPED, i.e. NOT tested -- see above)"
    print(f"\n{summary} -- {label}")
    return all_passed

In [ ]:
notebook_ns = {
    'config': config,
    'simulate_macro_paths': simulate_macro_paths,
    'build_feature_matrix': build_feature_matrix,
    'cascade_row': cascade_row,
    'build_transition_tensor': build_transition_tensor,
    'simulate_observed_counts': simulate_observed_counts,
    '_stage_log_odds_and_weight': _stage_log_odds_and_weight,
    'build_all_features': build_all_features,
    'fit_ols': fit_ols,
    'engine_sanity_check': engine_sanity_check,
    'validate_wls_reduces_to_ols': validate_wls_reduces_to_ols,
    'validate_softmax_and_chain': validate_softmax_and_chain,
    'compute_x_train_bounds': compute_x_train_bounds,
    'predict_scenario': predict_scenario,
    'build_apps_script': build_apps_script,
    'verify_apps_script': verify_apps_script,
    'model_weights_path': 'model_weights.json',
    'code_gs_path': 'sheets/Code.gs',
    'build_workbook': build_workbook,
    'verify_workbook': verify_workbook,
    'excel_path': 'credit_migration_scenario_calculator.xlsx',
}

passed = run_all(notebook_ns, 'Notebook (this Colab/Jupyter kernel, live)')
assert passed, 'stress suite failed inside the notebook -- see the FAIL lines above'